# 01 — ViT Training, Logit Caching, and Source-Bank Generation

This notebook prepares the ViT-B/16 branch of the Support-Aware Stream Calibration (SASC) experiments. It is part of the public artifact for the paper and is intended to document the full ViT data/logit-generation workflow.

## Purpose

This notebook covers:

- ImageNet-100 extraction and class mapping;
- selective ImageNet-C filtering to the same 100 classes;
- ViT-B/16 training with the frozen train/calibration split;
- clean calibration and validation logit caching;
- ImageNet-C target logit caching;
- source temperature scaling refit;
- target-oracle diagnostic temperature fitting;
- source-bank logit generation for downstream SASC analysis.

## Important notes

- This notebook was originally run in Google Colab using Google Drive paths. Update the path configuration cell before rerunning in another environment.
- ImageNet-100, ImageNet-C, cached logits, and checkpoints are not committed to the GitHub repository because they are large and/or subject to separate dataset terms.
- Target-oracle temperatures are diagnostic only because they use target labels. They are not deployable calibration methods.
- The downstream SASC/Frozen V3 analysis is performed in `02_vit_sasc_analysis_and_diagnostics.ipynb`.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import json
import tarfile
import zipfile
import random
import time
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import shutil
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights

PATHS = {
    "drive_root": "/content/drive/MyDrive",

    "vit_extra_root": "/content/drive/MyDrive/AML_paper_freeze_v3_vit_extra",

    "imagenet100_raw_zip": "/content/drive/MyDrive/imagenet 100/archive (1).zip",

    "imagenetc_tar_root": "/content/drive/MyDrive/imagenet-c",

    "imagenet100_local_root": "/content/data/imagenet100_extracted",
    "imagenetc_filtered_root": "/content/data/imagenet_c_100_filtered",

    "frozen_resnet_split_json": (
        "/content/drive/MyDrive/AML_final/splits/source_train_calib_split.json"
    ),
}

DRIVE_ROOT = Path(PATHS["drive_root"])

IMAGENET100_RAW_ZIP = Path(PATHS["imagenet100_raw_zip"])

IMAGENETC_ARCHIVE_ROOT = Path(PATHS["imagenetc_tar_root"])
IMAGENETC_ARCHIVES = {
    "blur":    IMAGENETC_ARCHIVE_ROOT / "blur.tar",
    "noise":   IMAGENETC_ARCHIVE_ROOT / "noise.tar",
    "weather": IMAGENETC_ARCHIVE_ROOT / "weather.tar",
}

LOCAL_IMAGENET100_EXTRACT_ROOT = Path(PATHS["imagenet100_local_root"])
LOCAL_IMAGENETC_FILTERED_ROOT = Path(PATHS["imagenetc_filtered_root"])

FROZEN_RESNET_SPLIT_JSON = Path(PATHS["frozen_resnet_split_json"])

VIT_EXTRA_ROOT = Path(PATHS["vit_extra_root"])

DIRS = {
    "root": VIT_EXTRA_ROOT,
    "configs": VIT_EXTRA_ROOT / "configs",
    "manifests": VIT_EXTRA_ROOT / "manifests",
    "splits": VIT_EXTRA_ROOT / "splits",
    "checkpoints": VIT_EXTRA_ROOT / "checkpoints",
    "logits_cache": VIT_EXTRA_ROOT / "logits_cache",
    "source_ts": VIT_EXTRA_ROOT / "source_ts",
    "target_oracle_diagnostics": VIT_EXTRA_ROOT / "target_oracle_diagnostics",
    "stream_methods": VIT_EXTRA_ROOT / "stream_methods",
    "tables": VIT_EXTRA_ROOT / "tables",
    "figures": VIT_EXTRA_ROOT / "figures",
    "audit": VIT_EXTRA_ROOT / "audit",
}

for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

LOCAL_IMAGENET100_EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_IMAGENETC_FILTERED_ROOT.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("Paths configured:")
print("=" * 70)
print(f"VIT_EXTRA_ROOT:                  {VIT_EXTRA_ROOT}")
print(f"LOCAL_IMAGENET100_EXTRACT_ROOT:  {LOCAL_IMAGENET100_EXTRACT_ROOT}")
print(f"LOCAL_IMAGENETC_FILTERED_ROOT:   {LOCAL_IMAGENETC_FILTERED_ROOT}")
print()
print("Checking required inputs on Drive:")
print(f"  ImageNet-100 raw zip:    exists={IMAGENET100_RAW_ZIP.exists()}")
print(f"    -> {IMAGENET100_RAW_ZIP}")
print()
print(f"  ImageNet-C raw tars:")
for group, path in IMAGENETC_ARCHIVES.items():
    print(f"    {group:8s} exists={path.exists()}  -> {path}")
print()
print(f"  Frozen ResNet split JSON: exists={FROZEN_RESNET_SPLIT_JSON.exists()}")
print(f"    -> {FROZEN_RESNET_SPLIT_JSON}")

# Hard requirements
assert IMAGENET100_RAW_ZIP.exists(), f"Missing ImageNet-100 raw zip: {IMAGENET100_RAW_ZIP}"

for group, path in IMAGENETC_ARCHIVES.items():
    assert path.exists(), (
        f"Missing ImageNet-C {group} tar: {path}\n"
        f"Expected at {IMAGENETC_ARCHIVE_ROOT}"
    )

assert FROZEN_RESNET_SPLIT_JSON.exists(), (
    f"Missing frozen ResNet split: {FROZEN_RESNET_SPLIT_JSON}\n"
    f"This is required to reuse the same train/calib split as the ResNets."
)

print()
print("Cell 1 OK. Using raw archives only — no prepared tars.")

**Fixed protocol and frozen ViT evaluation plan**

In [ ]:
SEEDS = [1, 2, 3]

CORRUPTIONS = [
    "brightness",
    "defocus_blur",
    "glass_blur",
    "motion_blur",
    "zoom_blur",
    "gaussian_noise",
    "shot_noise",
    "impulse_noise",
    "fog",
    "snow",
]

SEVERITIES = [1, 2, 3, 4, 5]

# Map each corruption to the tar archive it lives in.
# Used by selective extraction to know which tar to open for each corruption.
CORRUPTION_TO_ARCHIVE_GROUP = {
    "defocus_blur":   "blur",
    "glass_blur":     "blur",
    "motion_blur":    "blur",
    "zoom_blur":      "blur",
    "gaussian_noise": "noise",
    "shot_noise":     "noise",
    "impulse_noise":  "noise",
    "brightness":     "weather",
    "fog":            "weather",
    "snow":           "weather",
}

FROZEN_V3_CONFIG = {
    "method": "safe_stream_entropy_v3_capped_extrapolation",
    "n_entropy_bins": 8,
    "support_margin_std": 0.25,
    "t_max_absolute": 1.45,
    "t_max_additive": 0.35,
    "max_extrapolation_slope": 0.75,
    "lower_fallback_to_source_ts": True,
    "fit_data": "source_bank_only",
    "target_labels_used_for_fit": False,
    "target_metrics_used_for_hyperparameter_selection": False,
}

EXTRA_ARCHITECTURE_FREEZE = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "extra_architecture": "vit_b_16",
    "architecture_family": "vision_transformer",
    "pretrained_weights": "torchvision.models.ViT_B_16_Weights.IMAGENET1K_V1",
    "purpose": "Test whether stream-entropy calibration findings generalize beyond ResNet/CNN backbones.",
    "dataset": "ImageNet-100 source + selected ImageNet-C corruptions filtered to ImageNet-100 classes",
    "data_source": "raw archives (archive (1).zip + blur/noise/weather.tar)",
    "seeds": SEEDS,
    "corruptions": CORRUPTIONS,
    "severities": SEVERITIES,
    "v3_config_changed": False,
    "v3_config": FROZEN_V3_CONFIG,
    "pre_committed_checks": [
        "Does entropy-mean still improve over source TS?",
        "Do rich stream descriptors still overfit or hurt?",
        "Does frozen V3 improve over source TS?",
        "Does frozen V3 differ from V2 or collapse to V2?",
        "Does oracle target temperature remain corruption/severity dependent?",
        "What fraction of target streams are inside support vs high extrapolation?",
    ],
    "oracle_status": "Target-oracle TS is diagnostic only because it uses target labels.",
    "honesty_note": "No V3 hyperparameters will be changed after observing ViT target results.",
}

freeze_path = DIRS["configs"] / "extra_architecture_vit_freeze_plan.json"

with open(freeze_path, "w") as f:
    json.dump(EXTRA_ARCHITECTURE_FREEZE, f, indent=2)

print("Saved frozen ViT evaluation plan to:")
print(freeze_path)

**Helper functions**

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def looks_like_synset(name: str) -> bool:
    """
    ImageNet synset folder names usually look like n01440764.
    """
    return len(name) == 9 and name.startswith("n") and name[1:].isdigit()


def has_images(path: Path) -> bool:
    for p in path.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            return True
    return False


def count_images_under(path: Path) -> int:
    count = 0
    for p in path.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            count += 1
    return count


def print_tree(root: Path, max_depth: int = 3, max_items_per_level: int = 20):
    root = Path(root)
    print(f"\nTREE: {root}")

    def _walk(path: Path, depth: int):
        if depth > max_depth:
            return

        try:
            items = sorted(
                list(path.iterdir()),
                key=lambda x: (not x.is_dir(), x.name.lower())
            )
        except Exception as e:
            print("  " * depth + f"[cannot read] {path}: {e}")
            return

        shown = items[:max_items_per_level]

        for item in shown:
            suffix = "/" if item.is_dir() else ""
            print("  " * depth + f"- {item.name}{suffix}")
            if item.is_dir():
                _walk(item, depth + 1)

        if len(items) > max_items_per_level:
            print("  " * depth + f"... ({len(items) - max_items_per_level} more items)")

    _walk(root, 0)


def is_imagefolder_candidate(path: Path, min_class_dirs: int = 20) -> bool:
    if not path.is_dir():
        return False

    subdirs = [p for p in path.iterdir() if p.is_dir()]

    if len(subdirs) < min_class_dirs:
        return False

    good_subdirs = 0

    for sd in subdirs[:min(120, len(subdirs))]:
        if has_images(sd):
            good_subdirs += 1

    return good_subdirs >= min_class_dirs


def find_imagefolder_candidates(root: Path, min_class_dirs: int = 20):
    candidates = []

    if is_imagefolder_candidate(root, min_class_dirs=min_class_dirs):
        candidates.append(root)

    for p in root.rglob("*"):
        if p.is_dir() and is_imagefolder_candidate(p, min_class_dirs=min_class_dirs):
            candidates.append(p)

    unique = []
    seen = set()

    for c in candidates:
        c_str = str(c)
        if c_str not in seen:
            seen.add(c_str)
            unique.append(c)

    return unique


def choose_best_imagenet100_root(candidates):
    """
    Prefer the folder with the largest number of synset-like class folders.
    Prefer train over val when tied.
    """
    scored = []

    for c in candidates:
        class_dirs = [p for p in c.iterdir() if p.is_dir()]
        synset_dirs = [p for p in class_dirs if looks_like_synset(p.name)]
        image_class_dirs = [p for p in synset_dirs if has_images(p)]

        score = (
            len(image_class_dirs),
            len(synset_dirs),
            1 if "train" in str(c).lower() else 0,
            -1 if "val" in str(c).lower() else 0,
            -len(str(c)),
        )

        scored.append((score, c, len(image_class_dirs), len(synset_dirs), len(class_dirs)))

    scored = sorted(scored, reverse=True)
    return scored


print("Helper functions ready.")

**Extract ImageNet-100 from raw zip**

In [ ]:
imagenet100_marker = LOCAL_IMAGENET100_EXTRACT_ROOT / ".imagenet100_extracted_done"

if imagenet100_marker.exists():
    print(f"[SKIP] ImageNet-100 already extracted to {LOCAL_IMAGENET100_EXTRACT_ROOT}")
else:
    print(f"[EXTRACT] {IMAGENET100_RAW_ZIP}")
    print(f"  -> {LOCAL_IMAGENET100_EXTRACT_ROOT}")
    print("  (this takes ~10-15 minutes for ~15GB zip)")
    with zipfile.ZipFile(IMAGENET100_RAW_ZIP, "r") as zf:
        zf.extractall(LOCAL_IMAGENET100_EXTRACT_ROOT)
    imagenet100_marker.write_text(datetime.now().isoformat())
    print("[DONE] ImageNet-100 extracted.")

# Inspect the extracted tree
print_tree(LOCAL_IMAGENET100_EXTRACT_ROOT, max_depth=3, max_items_per_level=15)

# Find the ImageFolder root
imagenet100_candidates = find_imagefolder_candidates(
    LOCAL_IMAGENET100_EXTRACT_ROOT,
    min_class_dirs=50,
)

print("\nPotential ImageNet-100 ImageFolder candidates:")
for i, p in enumerate(imagenet100_candidates):
    print(f"{i}: {p}")

scored_candidates = choose_best_imagenet100_root(imagenet100_candidates)

print("\nScored candidates (image_synsets, synsets, total_dirs):")
for i, (score, path, n_image_synsets, n_synsets, n_dirs) in enumerate(scored_candidates[:10]):
    print(
        f"{i}: {path} | image_synsets={n_image_synsets} | "
        f"synsets={n_synsets} | total_dirs={n_dirs} | score={score}"
    )

if not scored_candidates:
    raise RuntimeError("No valid ImageNet-100 ImageFolder root found. Inspect tree output.")

SOURCE_IMAGEFOLDER_ROOT = scored_candidates[0][1]

IMAGENET100_SYNSETS = sorted([
    p.name
    for p in SOURCE_IMAGEFOLDER_ROOT.iterdir()
    if p.is_dir() and looks_like_synset(p.name) and has_images(p)
])

print("\nSelected SOURCE_IMAGEFOLDER_ROOT:")
print(SOURCE_IMAGEFOLDER_ROOT)

print(f"Detected ImageNet-100 classes: {len(IMAGENET100_SYNSETS)}")
print(f"First 10 classes: {IMAGENET100_SYNSETS[:10]}")

assert len(IMAGENET100_SYNSETS) == 100, (
    f"Expected exactly 100 ImageNet-100 classes, got {len(IMAGENET100_SYNSETS)}.\n"
    f"Inspect the tree output above. The raw zip may be wrong or extract failed."
)

# Sanity check: count total images
total_images = sum(
    count_images_under(SOURCE_IMAGEFOLDER_ROOT / s)
    for s in IMAGENET100_SYNSETS
)
print(f"Total ImageNet-100 train images: {total_images}")

# Warn if it doesn't match what the frozen split expects (126689)
if total_images != 126689:
    print(f"\nWARNING: total images = {total_images}, but frozen ResNet split expects n_total=126689.")
    print("The split assertion in cell 13 will catch this if it's still a mismatch.")
    print("Inspect the tree output and the candidates above to see if cell 4 picked the wrong root.")
else:
    print("Image count matches frozen ResNet split (126689). Good.")

**Save ImageNet-100 class mapping**

In [ ]:
class_to_idx = {synset: idx for idx, synset in enumerate(IMAGENET100_SYNSETS)}
idx_to_class = {idx: synset for synset, idx in class_to_idx.items()}

class_mapping = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "source_imagefolder_root": str(SOURCE_IMAGEFOLDER_ROOT),
    "num_classes": len(IMAGENET100_SYNSETS),
    "classes": IMAGENET100_SYNSETS,
    "class_to_idx": class_to_idx,
    "idx_to_class": idx_to_class,
}

class_mapping_path = DIRS["manifests"] / "imagenet100_class_mapping.json"

with open(class_mapping_path, "w") as f:
    json.dump(class_mapping, f, indent=2)

print(f"Saved class mapping to: {class_mapping_path}")

# Peek into each ImageNet-C tar to confirm structure
print("\n" + "=" * 70)
print("Peeking inside ImageNet-C tars (first 15 members each):")
print("=" * 70)
for group, path in IMAGENETC_ARCHIVES.items():
    print(f"\n  {group} -> {path}")
    with tarfile.open(path, "r") as tf:
        for i, member in enumerate(tf):
            if i >= 15:
                break
            print(f"    {member.name}")

**Selective extraction from raw ImageNet-C tars**

In [ ]:
# ============================================================
# Extract/filter original ImageNet-C tar files to ImageNet-C-100
# ============================================================

imagenetc_marker = LOCAL_IMAGENETC_FILTERED_ROOT / ".imagenet_c_100_extracted_done"

selected_synsets = set(IMAGENET100_SYNSETS)

if imagenetc_marker.exists():
    print("[SKIP] Filtered ImageNet-C-100 already extracted.")
else:
    print("[RUN] Stream-filtering original ImageNet-C tar files...")

    if LOCAL_IMAGENETC_FILTERED_ROOT.exists():
        shutil.rmtree(LOCAL_IMAGENETC_FILTERED_ROOT)
    LOCAL_IMAGENETC_FILTERED_ROOT.mkdir(parents=True, exist_ok=True)

    total_extracted = 0
    total_skipped = 0

    for archive_name, archive_path in IMAGENETC_ARCHIVES.items():
        archive_path = Path(archive_path)
        assert archive_path.exists(), f"Missing ImageNet-C archive: {archive_path}"

        print(f"\nStreaming {archive_name}: {archive_path}")

        extracted = 0
        skipped = 0

        with tarfile.open(archive_path, mode="r|*") as tar:
            for member in tar:
                if not member.isfile():
                    continue

                parts = member.name.split("/")

                # Expected: corruption/severity/synset/image.JPEG
                if len(parts) < 4:
                    skipped += 1
                    continue

                synset = parts[2]

                if synset in selected_synsets:
                    tar.extract(member, path=LOCAL_IMAGENETC_FILTERED_ROOT, filter="data")
                    extracted += 1
                else:
                    skipped += 1

                if (extracted + skipped) % 50000 == 0:
                    print(
                        f"  processed={extracted + skipped:,} | "
                        f"kept={extracted:,} | skipped={skipped:,}"
                    )

        print(f"[DONE] {archive_name}: kept={extracted:,}, skipped={skipped:,}")

        total_extracted += extracted
        total_skipped += skipped

    imagenetc_marker.write_text(datetime.now().isoformat())

    print("\n[DONE] Filtered ImageNet-C-100 extracted.")
    print(f"Total kept:    {total_extracted:,}")
    print(f"Total skipped: {total_skipped:,}")

# Print what we got
print(f"\nContents of {LOCAL_IMAGENETC_FILTERED_ROOT}:")
top_items = sorted(LOCAL_IMAGENETC_FILTERED_ROOT.iterdir())

for item in top_items[:20]:
    kind = "DIR " if item.is_dir() else "FILE"
    print(f"  {kind}  {item.name}")

if len(top_items) > 20:
    print(f"  ... ({len(top_items) - 20} more)")

**Validate filtered ImageNet-C**

In [ ]:
selected_synsets = set(IMAGENET100_SYNSETS)

imagenetc_extraction_summary = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "filtered_root": str(LOCAL_IMAGENETC_FILTERED_ROOT),
    "selected_synset_count": len(selected_synsets),
    "selected_synsets": sorted(list(selected_synsets)),
    "corruptions": CORRUPTIONS,
    "severities": SEVERITIES,
    "source": "selectively extracted from raw ImageNet-C tar archives",
    "raw_tars": {k: str(v) for k, v in IMAGENETC_ARCHIVES.items()},
}

condition_validation_rows = []

for corruption in CORRUPTIONS:
    for severity in SEVERITIES:
        condition_root = LOCAL_IMAGENETC_FILTERED_ROOT / corruption / str(severity)

        exists = condition_root.exists()

        if exists:
            class_dirs = sorted([
                p.name
                for p in condition_root.iterdir()
                if p.is_dir()
            ])
        else:
            class_dirs = []

        missing_classes = sorted(list(selected_synsets - set(class_dirs)))
        extra_classes = sorted(list(set(class_dirs) - selected_synsets))

        image_count = 0
        if exists:
            for synset in class_dirs:
                image_count += count_images_under(condition_root / synset)

        row = {
            "corruption": corruption,
            "severity": severity,
            "condition_root": str(condition_root),
            "exists": exists,
            "num_class_dirs": len(class_dirs),
            "num_missing_classes": len(missing_classes),
            "num_extra_classes": len(extra_classes),
            "num_images": image_count,
            "first_missing_classes": missing_classes[:10],
            "first_extra_classes": extra_classes[:10],
        }

        condition_validation_rows.append(row)

print("Filtered ImageNet-C validation:")

for row in condition_validation_rows:
    print(
        f"{row['corruption']:15s} sev={row['severity']} | "
        f"exists={row['exists']} | "
        f"class_dirs={row['num_class_dirs']} | "
        f"missing={row['num_missing_classes']} | "
        f"extra={row['num_extra_classes']} | "
        f"images={row['num_images']}"
    )

bad_conditions = [
    row for row in condition_validation_rows
    if (not row["exists"]) or row["num_class_dirs"] != 100 or row["num_missing_classes"] != 0
]

print("\nNumber of bad/incomplete conditions:", len(bad_conditions))

if bad_conditions:
    print("First bad conditions:")
    for row in bad_conditions[:10]:
        print(row)


imagenetc_extraction_summary["condition_validation"] = condition_validation_rows

imagenetc_manifest_path = DIRS["manifests"] / "imagenetc_100_filtered_extraction_manifest.json"

with open(imagenetc_manifest_path, "w") as f:
    json.dump(imagenetc_extraction_summary, f, indent=2)

stage0_manifest = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "vit_extra_root": str(VIT_EXTRA_ROOT),
    "source_imagefolder_root": str(SOURCE_IMAGEFOLDER_ROOT),
    "imagenet100_source": {
        "raw_zip": str(IMAGENET100_RAW_ZIP),
        "raw_zip_exists": IMAGENET100_RAW_ZIP.exists(),
        "local_extract_root": str(LOCAL_IMAGENET100_EXTRACT_ROOT),
    },
    "imagenet100_num_classes": len(IMAGENET100_SYNSETS),
    "imagenet100_classes": IMAGENET100_SYNSETS,
    "imagenetc_source": {
        "raw_tars": {k: str(v) for k, v in IMAGENETC_ARCHIVES.items()},
        "all_tars_exist": all(v.exists() for v in IMAGENETC_ARCHIVES.values()),
        "local_filtered_root": str(LOCAL_IMAGENETC_FILTERED_ROOT),
    },
    "frozen_resnet_split_json": str(FROZEN_RESNET_SPLIT_JSON),
    "corruptions": CORRUPTIONS,
    "severities": SEVERITIES,
    "seeds": SEEDS,
    "frozen_vit_plan_path": str(freeze_path),
    "class_mapping_path": str(class_mapping_path),
    "imagenetc_filtered_manifest_path": str(imagenetc_manifest_path),
}

stage0_manifest_path = DIRS["manifests"] / "stage0_setup_manifest.json"

with open(stage0_manifest_path, "w") as f:
    json.dump(stage0_manifest, f, indent=2)

print("Saved ImageNet-C filtered manifest:")
print(imagenetc_manifest_path)

print("\nSaved Stage 0 setup manifest:")
print(stage0_manifest_path)

print("\n========== STAGE 0 SUMMARY ==========")
print("Source ImageFolder root:", SOURCE_IMAGEFOLDER_ROOT)
print("Number of ImageNet-100 classes:", len(IMAGENET100_SYNSETS))
print("Filtered ImageNet-C root:", LOCAL_IMAGENETC_FILTERED_ROOT)
print("Number of bad/incomplete ImageNet-C conditions:", len(bad_conditions))

if len(bad_conditions) == 0:
    print("Stage 0 completed successfully.")
else:
    print("Stage 0 completed, but some conditions are incomplete. Inspect bad_conditions before training.")

**Torch imports and training configuration**

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    # Quick capability check (Colab T4=sm_75, L4=sm_89, A100=sm_80, V100=sm_70)
    major, minor = torch.cuda.get_device_capability(0)
    print(f"  CUDA capability: sm_{major}{minor}")

ARCH_NAME = "vit_b_16"
NUM_CLASSES = 100

TRAIN_CONFIG = {
    "architecture": ARCH_NAME,
    "pretrained": "ImageNet-1k",
    "num_classes": NUM_CLASSES,
    "image_size": 224,
    "train_calib_split": {
        "source": "frozen ResNet split loaded from AML_final/splits/source_train_calib_split.json",
        "train_fraction_target": 0.90,
        "calib_fraction_target": 0.10,
        "stratified": True,
        "reused_for_protocol_parity": True,
    },
    "epochs": 10,
    "batch_size": 32,
    "batch_size_deviation_note": (
        "ResNet recipe used batch_size=128. ViT-B/16 cannot fit batch_size=128 "
        "on a 16GB GPU (Colab T4/V100). Using batch_size=32 with AMP. "
        "This is a documented deviation from the ResNet recipe."
    ),
    "num_workers": 2,
    "optimizer": "SGD",
    "lr": 0.01,
    "momentum": 0.9,
    "weight_decay": 1e-4,
    "scheduler": "CosineAnnealingLR",
    "amp": True,
    "seeds": SEEDS,
    "recipe_match_to_resnet": (
        "SGD lr=0.01 + momentum=0.9 + wd=1e-4 + CosineAnnealingLR(T_max=10) + 10 epochs "
        "matched to ResNet protocol. Pre-registered: no optimizer/LR changes based on outcome."
    ),
}

train_config_path = DIRS["configs"] / "stage1_vit_training_config.json"

with open(train_config_path, "w") as f:
    json.dump(TRAIN_CONFIG, f, indent=2)

print("\nSaved Stage 1 training config:")
print(train_config_path)

**Reproducibility helpers**

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_generator(seed: int):
    g = torch.Generator()
    g.manual_seed(seed)
    return g


print("Seed helpers ready.")

**train and clean validation ImageNet-100 roots**

In [ ]:
print("SOURCE_IMAGEFOLDER_ROOT from Stage 0:")
print(SOURCE_IMAGEFOLDER_ROOT)

def candidate_has_same_classes(candidate_root: Path, expected_classes):
    if not candidate_root.exists() or not candidate_root.is_dir():
        return False

    class_dirs = sorted([
        p.name for p in candidate_root.iterdir()
        if p.is_dir() and looks_like_synset(p.name)
    ])

    return class_dirs == sorted(expected_classes)


def find_clean_val_root(source_root: Path, expected_classes):
    """
    Try to find a separate validation ImageFolder root near the selected source root.
    """
    source_root = Path(source_root)

    possible_names = [
        "val",
        "validation",
        "valid",
        "test",
        "imagenet100_val",
        "imagenet-100-val",
    ]

    candidates = []

    # siblings
    for name in possible_names:
        candidates.append(source_root.parent / name)

    # parent siblings
    if source_root.parent is not None:
        for name in possible_names:
            candidates.append(source_root.parent.parent / name)

    # search under extracted root
    for p in LOCAL_IMAGENET100_EXTRACT_ROOT.rglob("*"):
        if p.is_dir() and any(name in p.name.lower() for name in possible_names):
            candidates.append(p)

    # unique
    unique = []
    seen = set()
    for c in candidates:
        c = Path(c)
        if str(c) not in seen:
            seen.add(str(c))
            unique.append(c)

    valid = []
    for c in unique:
        if candidate_has_same_classes(c, expected_classes):
            valid.append(c)

    return valid


TRAIN_IMAGEFOLDER_ROOT = Path(SOURCE_IMAGEFOLDER_ROOT)
VAL_CANDIDATES = find_clean_val_root(TRAIN_IMAGEFOLDER_ROOT, IMAGENET100_SYNSETS)

print("\nTRAIN_IMAGEFOLDER_ROOT:")
print(TRAIN_IMAGEFOLDER_ROOT)

print("\nValidation candidates:")
for i, p in enumerate(VAL_CANDIDATES):
    print(f"{i}: {p}")

# If a valid separate val folder is found, use the first one.
# If not found, set CLEAN_VAL_IMAGEFOLDER_ROOT manually in this cell.
if len(VAL_CANDIDATES) > 0:
    CLEAN_VAL_IMAGEFOLDER_ROOT = VAL_CANDIDATES[0]
else:
    CLEAN_VAL_IMAGEFOLDER_ROOT = None

print("\nSelected CLEAN_VAL_IMAGEFOLDER_ROOT:")
print(CLEAN_VAL_IMAGEFOLDER_ROOT)

if CLEAN_VAL_IMAGEFOLDER_ROOT is None:
    print("\nWARNING: No separate val folder detected automatically.")
    print("If your archive has a val folder, set CLEAN_VAL_IMAGEFOLDER_ROOT manually before continuing.")

**Manual validation-root override if needed**

In [ ]:
MANUAL_CLEAN_VAL_ROOT = None

if MANUAL_CLEAN_VAL_ROOT is not None:
    MANUAL_CLEAN_VAL_ROOT = Path(MANUAL_CLEAN_VAL_ROOT)
    assert candidate_has_same_classes(MANUAL_CLEAN_VAL_ROOT, IMAGENET100_SYNSETS), (
        f"Manual val root does not match ImageNet-100 classes: {MANUAL_CLEAN_VAL_ROOT}"
    )
    CLEAN_VAL_IMAGEFOLDER_ROOT = MANUAL_CLEAN_VAL_ROOT

assert TRAIN_IMAGEFOLDER_ROOT.exists(), f"Missing train root: {TRAIN_IMAGEFOLDER_ROOT}"
assert CLEAN_VAL_IMAGEFOLDER_ROOT is not None, "CLEAN_VAL_IMAGEFOLDER_ROOT is still None."
assert CLEAN_VAL_IMAGEFOLDER_ROOT.exists(), f"Missing clean val root: {CLEAN_VAL_IMAGEFOLDER_ROOT}"

print("Final TRAIN_IMAGEFOLDER_ROOT:")
print(TRAIN_IMAGEFOLDER_ROOT)

print("\nFinal CLEAN_VAL_IMAGEFOLDER_ROOT:")
print(CLEAN_VAL_IMAGEFOLDER_ROOT)

**Transforms and datasets**

In [ ]:
assert TRAIN_IMAGEFOLDER_ROOT != CLEAN_VAL_IMAGEFOLDER_ROOT, (
    f"TRAIN and VAL roots are the same: {TRAIN_IMAGEFOLDER_ROOT}\n"
    f"Cell 10/11 logic must have collapsed them."
)
assert "train" in str(TRAIN_IMAGEFOLDER_ROOT).lower(), (
    f"TRAIN_IMAGEFOLDER_ROOT should point at a 'train' folder, got: {TRAIN_IMAGEFOLDER_ROOT}"
)
print(f"TRAIN_IMAGEFOLDER_ROOT: {TRAIN_IMAGEFOLDER_ROOT}")
print(f"CLEAN_VAL_IMAGEFOLDER_ROOT: {CLEAN_VAL_IMAGEFOLDER_ROOT}")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1,
    ),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Dataset used for train/calib split
full_train_dataset = datasets.ImageFolder(
    root=str(TRAIN_IMAGEFOLDER_ROOT),
    transform=train_transform,
)

# Same root but eval transform for calibration logits later
full_train_eval_transform_dataset = datasets.ImageFolder(
    root=str(TRAIN_IMAGEFOLDER_ROOT),
    transform=eval_transform,
)

clean_val_dataset = datasets.ImageFolder(
    root=str(CLEAN_VAL_IMAGEFOLDER_ROOT),
    transform=eval_transform,
)

expected_classes = sorted(IMAGENET100_SYNSETS)

assert full_train_dataset.classes == expected_classes, "Train dataset class order mismatch."
assert full_train_eval_transform_dataset.classes == expected_classes, "Train eval-transform dataset class order mismatch."
assert clean_val_dataset.classes == expected_classes, "Clean val dataset class order mismatch."

print("Train images:", len(full_train_dataset))
print("Clean val images:", len(clean_val_dataset))
print("Num classes:", len(full_train_dataset.classes))
print("First 10 classes:", full_train_dataset.classes[:10])

**Load frozen ResNet train/calib split**

In [ ]:
assert FROZEN_RESNET_SPLIT_JSON.exists(), f"Missing frozen ResNet split: {FROZEN_RESNET_SPLIT_JSON}"

with open(FROZEN_RESNET_SPLIT_JSON, "r") as f:
    frozen_split_payload = json.load(f)

print("Loaded frozen ResNet split from:")
print(f"  {FROZEN_RESNET_SPLIT_JSON}")
print(f"  keys: {list(frozen_split_payload.keys())}")

# The expected keys may vary; print and assert.
print(f"\nSplit payload contents:")
for k, v in frozen_split_payload.items():
    if isinstance(v, list):
        print(f"  {k}: list of length {len(v)}")
    elif isinstance(v, dict):
        print(f"  {k}: dict with {len(v)} keys")
    else:
        print(f"  {k}: {v}")

# Extract indices
assert "train_idx" in frozen_split_payload or "train_indices" in frozen_split_payload, (
    f"Split JSON has neither 'train_idx' nor 'train_indices' key. "
    f"Keys are: {list(frozen_split_payload.keys())}"
)
assert "calib_idx" in frozen_split_payload or "calib_indices" in frozen_split_payload, (
    f"Split JSON has neither 'calib_idx' nor 'calib_indices' key. "
    f"Keys are: {list(frozen_split_payload.keys())}"
)

train_indices = np.array(
    frozen_split_payload.get("train_idx") or frozen_split_payload.get("train_indices"),
    dtype=np.int64,
)
calib_indices = np.array(
    frozen_split_payload.get("calib_idx") or frozen_split_payload.get("calib_indices"),
    dtype=np.int64,
)

print(f"\nLoaded indices:")
print(f"  train: {len(train_indices)}")
print(f"  calib: {len(calib_indices)}")
print(f"  train + calib = {len(train_indices) + len(calib_indices)}")
print(f"  dataset size:   {len(full_train_dataset)}")

# Diagnostic before strict assertions (saves debugging time if the assert fails)
expected_total = frozen_split_payload.get("n_total")
if expected_total is not None:
    print(f"  n_total in JSON: {expected_total}")

# ---- Strict assertions ----
# (1) Indices cover the full dataset
assert len(train_indices) + len(calib_indices) == len(full_train_dataset), (
    f"Split coverage mismatch: train ({len(train_indices)}) + calib ({len(calib_indices)}) "
    f"= {len(train_indices) + len(calib_indices)} != dataset size ({len(full_train_dataset)}).\n"
    f"This means the frozen split was built on a different ImageNet-100 dataset size.\n"
    f"Cannot proceed without breaking protocol parity."
)

# (2) No overlap
assert len(set(train_indices.tolist()) & set(calib_indices.tolist())) == 0, (
    "Train and calib indices overlap. Split is malformed."
)

# (3) Indices are in range
assert train_indices.min() >= 0 and train_indices.max() < len(full_train_dataset), (
    f"Train indices out of range: min={train_indices.min()}, max={train_indices.max()}, "
    f"dataset_size={len(full_train_dataset)}"
)
assert calib_indices.min() >= 0 and calib_indices.max() < len(full_train_dataset), (
    f"Calib indices out of range: min={calib_indices.min()}, max={calib_indices.max()}, "
    f"dataset_size={len(full_train_dataset)}"
)

# (4) All 100 classes present in both partitions
targets = np.array(full_train_dataset.targets)
train_classes = set(targets[train_indices].tolist())
calib_classes = set(targets[calib_indices].tolist())

assert len(train_classes) == NUM_CLASSES, (
    f"Train split missing {NUM_CLASSES - len(train_classes)} classes."
)
assert len(calib_classes) == NUM_CLASSES, (
    f"Calib split missing {NUM_CLASSES - len(calib_classes)} classes."
)

# (5) Class coverage in calib is at least 1 per class
calib_class_counts = np.bincount(targets[calib_indices], minlength=NUM_CLASSES)
assert calib_class_counts.min() > 0, (
    f"Some classes have 0 calib examples. Min count: {calib_class_counts.min()}"
)

print(f"\nSplit validation passed.")
print(f"  Calib per-class min: {calib_class_counts.min()}")
print(f"  Calib per-class max: {calib_class_counts.max()}")
print(f"  Calib per-class mean: {calib_class_counts.mean():.1f}")

# ---- Save a copy to ViT outputs for audit ----
split_manifest_path = DIRS["splits"] / "vit_train_calib_split_FROZEN_FROM_RESNET.json"

split_manifest = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "purpose": "ViT-B/16 uses the frozen ResNet train/calib split for protocol parity.",
    "loaded_from": str(FROZEN_RESNET_SPLIT_JSON),
    "train_imagefolder_root": str(TRAIN_IMAGEFOLDER_ROOT),
    "clean_val_imagefolder_root": str(CLEAN_VAL_IMAGEFOLDER_ROOT),
    "num_total_train_root_images": int(len(full_train_dataset)),
    "num_train_split": int(len(train_indices)),
    "num_calib_split": int(len(calib_indices)),
    "num_clean_val": int(len(clean_val_dataset)),
    "train_fraction": float(len(train_indices) / len(full_train_dataset)),
    "calib_fraction": float(len(calib_indices) / len(full_train_dataset)),
    "stratified": True,
    "classes": full_train_dataset.classes,
    "frozen_split_source_metadata": {
        k: v for k, v in frozen_split_payload.items()
        if k not in ("train_idx", "calib_idx", "train_indices", "calib_indices")
    },
    "train_indices": train_indices.tolist(),
    "calib_indices": calib_indices.tolist(),
}

with open(split_manifest_path, "w") as f:
    json.dump(split_manifest, f, indent=2)

# Also copy the original ResNet split for audit
shutil.copy2(
    FROZEN_RESNET_SPLIT_JSON,
    DIRS["splits"] / "source_train_calib_split_FROM_RESNET_ORIGINAL.json"
)

print(f"\nSaved ViT split manifest:")
print(f"  {split_manifest_path}")
print(f"Copied original ResNet split for audit:")
print(f"  {DIRS['splits'] / 'source_train_calib_split_FROM_RESNET_ORIGINAL.json'}")

**Build ViT-B/16 model**

In [ ]:
def build_vit_b16_100class():
    weights = ViT_B_16_Weights.IMAGENET1K_V1
    model = vit_b_16(weights=weights)

    in_features = model.heads.head.in_features
    model.heads.head = nn.Linear(in_features, NUM_CLASSES)

    return model


# Quick model test
model_test = build_vit_b16_100class()
model_test.eval()

dummy = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    out = model_test(dummy)

print("Dummy output shape:", out.shape)

assert out.shape == (2, NUM_CLASSES)

del model_test
torch.cuda.empty_cache()

**Metric and evaluation helpers**

In [ ]:
@torch.no_grad()
def evaluate_model(model, loader, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    all_logits = []
    all_labels = []

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = F.cross_entropy(logits, y, reduction="sum")

        pred = logits.argmax(dim=1)
        correct = pred.eq(y).sum().item()

        total_loss += loss.item()
        total_correct += correct
        total_samples += y.size(0)

        all_logits.append(logits.detach().cpu())
        all_labels.append(y.detach().cpu())

    mean_nll = total_loss / max(total_samples, 1)
    accuracy = total_correct / max(total_samples, 1)

    logits = torch.cat(all_logits, dim=0)
    labels = torch.cat(all_labels, dim=0)

    return {
        "accuracy": float(accuracy),
        "nll": float(mean_nll),
        "num_samples": int(total_samples),
        "logits": logits,
        "labels": labels,
    }


def save_checkpoint(path, model, optimizer, scheduler, epoch, seed, metrics, config):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    torch.save(
        {
            "architecture": ARCH_NAME,
            "seed": int(seed),
            "epoch": int(epoch),
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "metrics": metrics,
            "config": config,
            "classes": IMAGENET100_SYNSETS,
            "class_to_idx": class_to_idx,
        },
        path,
    )


def save_logits_npz(path, logits, labels, metadata):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    np.savez_compressed(
        path,
        logits=logits.numpy().astype(np.float32),
        labels=labels.numpy().astype(np.int64),
        metadata=json.dumps(metadata),
    )


print("Metric/evaluation helpers ready.")

**Training function for one seed**

In [ ]:
def train_one_seed(seed: int):
    set_seed(seed)

    print(f"\n{'='*80}")
    print(f"Training {ARCH_NAME} | seed={seed}")
    print(f"{'='*80}")

    seed_dir = DIRS["checkpoints"] / ARCH_NAME / f"seed_{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)

    logits_seed_dir = DIRS["logits_cache"] / ARCH_NAME / f"seed_{seed}"
    logits_seed_dir.mkdir(parents=True, exist_ok=True)

    train_subset = Subset(full_train_dataset, train_indices)

    calib_subset_eval = Subset(full_train_eval_transform_dataset, calib_indices)

    # Dataloaders
    train_loader = DataLoader(
        train_subset,
        batch_size=TRAIN_CONFIG["batch_size"],
        shuffle=True,
        num_workers=TRAIN_CONFIG["num_workers"],
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=make_generator(seed),
    )

    calib_loader_eval = DataLoader(
        calib_subset_eval,
        batch_size=TRAIN_CONFIG["batch_size"],
        shuffle=False,
        num_workers=TRAIN_CONFIG["num_workers"],
        pin_memory=True,
    )

    clean_val_loader = DataLoader(
        clean_val_dataset,
        batch_size=TRAIN_CONFIG["batch_size"],
        shuffle=False,
        num_workers=TRAIN_CONFIG["num_workers"],
        pin_memory=True,
    )

    # Model
    model = build_vit_b16_100class()
    model = model.to(DEVICE)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=TRAIN_CONFIG["lr"],
        momentum=TRAIN_CONFIG["momentum"],
        weight_decay=TRAIN_CONFIG["weight_decay"],
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=TRAIN_CONFIG["epochs"],
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(TRAIN_CONFIG["amp"] and DEVICE.type == "cuda"),
    )

    best_acc = -1.0
    best_nll = float("inf")

    history = []

    for epoch in range(1, TRAIN_CONFIG["epochs"] + 1):
        model.train()

        start_time = time.time()

        running_loss = 0.0
        running_correct = 0
        running_total = 0

        for x, y in train_loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(
                "cuda",
                enabled=(TRAIN_CONFIG["amp"] and DEVICE.type == "cuda"),
            ):
                logits = model(x)
                loss = F.cross_entropy(logits, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            with torch.no_grad():
                pred = logits.argmax(dim=1)
                running_correct += pred.eq(y).sum().item()
                running_total += y.size(0)
                running_loss += loss.item() * y.size(0)

        scheduler.step()

        train_loss = running_loss / max(running_total, 1)
        train_acc = running_correct / max(running_total, 1)

        # Pre-registered divergence check
        if not math.isfinite(train_loss):
            raise RuntimeError(
                f"DIVERGENCE: train_loss is non-finite at seed={seed} epoch={epoch}.\n"
                f"Per pre-registered protocol, training is halted.\n"
                f"This is the documented failure mode of the matched ResNet recipe "
                f"applied to ViT-B/16. Do NOT switch optimizers based on this outcome."
            )
        if epoch == 1 and train_acc < 0.05:
            print(
                f"WARNING: epoch 1 train_acc={train_acc:.4f} is near chance (1/100=0.01).\n"
                f"SGD lr=0.01 may be too aggressive for ViT-B/16.\n"
                f"Continuing per pre-registered protocol; this will be documented "
                f"in the analysis as a recipe-incompatibility finding."
            )

        val_eval = evaluate_model(model, clean_val_loader, DEVICE)
        calib_eval = evaluate_model(model, calib_loader_eval, DEVICE)

        epoch_time = time.time() - start_time

        row = {
            "seed": int(seed),
            "epoch": int(epoch),
            "train_loss": float(train_loss),
            "train_accuracy": float(train_acc),
            "clean_val_accuracy": float(val_eval["accuracy"]),
            "clean_val_nll": float(val_eval["nll"]),
            "calib_accuracy": float(calib_eval["accuracy"]),
            "calib_nll": float(calib_eval["nll"]),
            "lr": float(optimizer.param_groups[0]["lr"]),
            "epoch_time_sec": float(epoch_time),
        }

        history.append(row)

        print(
            f"seed={seed} epoch={epoch:02d}/{TRAIN_CONFIG['epochs']} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_acc={val_eval['accuracy']:.4f} val_nll={val_eval['nll']:.4f} | "
            f"calib_nll={calib_eval['nll']:.4f} | "
            f"time={epoch_time:.1f}s"
        )

        # Save best by accuracy
        if val_eval["accuracy"] > best_acc:
            best_acc = val_eval["accuracy"]
            save_checkpoint(
                seed_dir / "best_by_acc.pt",
                model,
                optimizer,
                scheduler,
                epoch,
                seed,
                metrics={
                    "clean_val_accuracy": val_eval["accuracy"],
                    "clean_val_nll": val_eval["nll"],
                    "calib_accuracy": calib_eval["accuracy"],
                    "calib_nll": calib_eval["nll"],
                },
                config=TRAIN_CONFIG,
            )

        # Save best by NLL
        if val_eval["nll"] < best_nll:
            best_nll = val_eval["nll"]
            save_checkpoint(
                seed_dir / "best_by_nll.pt",
                model,
                optimizer,
                scheduler,
                epoch,
                seed,
                metrics={
                    "clean_val_accuracy": val_eval["accuracy"],
                    "clean_val_nll": val_eval["nll"],
                    "calib_accuracy": calib_eval["accuracy"],
                    "calib_nll": calib_eval["nll"],
                },
                config=TRAIN_CONFIG,
            )

    # Save final checkpoint
    final_val_eval = evaluate_model(model, clean_val_loader, DEVICE)
    final_calib_eval = evaluate_model(model, calib_loader_eval, DEVICE)

    save_checkpoint(
        seed_dir / "final.pt",
        model,
        optimizer,
        scheduler,
        TRAIN_CONFIG["epochs"],
        seed,
        metrics={
            "clean_val_accuracy": final_val_eval["accuracy"],
            "clean_val_nll": final_val_eval["nll"],
            "calib_accuracy": final_calib_eval["accuracy"],
            "calib_nll": final_calib_eval["nll"],
        },
        config=TRAIN_CONFIG,
    )

    # Save history
    history_df = pd.DataFrame(history)
    history_path = seed_dir / "training_history.csv"
    history_df.to_csv(history_path, index=False)

    print("Saved training history:", history_path)

    result = {
        "seed": int(seed),
        "seed_dir": str(seed_dir),
        "history_path": str(history_path),
        "best_acc": float(best_acc),
        "best_nll": float(best_nll),
    }

    # ============================================================
    # GPU memory hygiene before next seed
    # ============================================================
    # ViT-B/16 + optimizer state + AMP scaler ≈ 700MB resident.
    # Without explicit cleanup, seeds 2 and 3 may OOM on tight budgets.
    # ============================================================
    del model, optimizer, scheduler, scaler
    torch.cuda.empty_cache()
    import gc
    gc.collect()

    return result


print("Training function ready.")

**Train all seeds**

In [ ]:
stage1_train_rows = []

for seed in SEEDS:
    result = train_one_seed(seed)
    stage1_train_rows.append(result)

stage1_training_summary = pd.DataFrame(stage1_train_rows)

stage1_training_summary_path = DIRS["tables"] / "stage1_vit_training_summary.csv"
stage1_training_summary.to_csv(stage1_training_summary_path, index=False)

print("\nSaved Stage 1 training summary:")
print(stage1_training_summary_path)

display(stage1_training_summary)

**Load best_by_nll checkpoint helper**

In [ ]:
def load_vit_best_by_nll(seed: int):
    checkpoint_path = DIRS["checkpoints"] / ARCH_NAME / f"seed_{seed}" / "best_by_nll.pt"

    assert checkpoint_path.exists(), f"Missing checkpoint: {checkpoint_path}"

    model = build_vit_b16_100class()

    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])

    model = model.to(DEVICE)
    model.eval()

    return model, ckpt, checkpoint_path


print("Checkpoint loading helper ready.")

**Cache clean calibration and clean validation logits**

In [ ]:
stage1_logit_rows = []

for seed in SEEDS:
    print(f"\nCaching clean logits | seed={seed}")

    model, ckpt, checkpoint_path = load_vit_best_by_nll(seed)

    calib_subset_eval = Subset(full_train_eval_transform_dataset, calib_indices)

    calib_loader_eval = DataLoader(
        calib_subset_eval,
        batch_size=TRAIN_CONFIG["batch_size"],
        shuffle=False,
        num_workers=TRAIN_CONFIG["num_workers"],
        pin_memory=True,
    )

    clean_val_loader = DataLoader(
        clean_val_dataset,
        batch_size=TRAIN_CONFIG["batch_size"],
        shuffle=False,
        num_workers=TRAIN_CONFIG["num_workers"],
        pin_memory=True,
    )

    calib_eval = evaluate_model(model, calib_loader_eval, DEVICE)
    val_eval = evaluate_model(model, clean_val_loader, DEVICE)

    seed_logits_dir = DIRS["logits_cache"] / ARCH_NAME / f"seed_{seed}"
    seed_logits_dir.mkdir(parents=True, exist_ok=True)

    calib_logits_path = seed_logits_dir / "calib_logits.npz"
    val_logits_path = seed_logits_dir / "val_logits_clean.npz"

    save_logits_npz(
        calib_logits_path,
        calib_eval["logits"],
        calib_eval["labels"],
        metadata={
            "architecture": ARCH_NAME,
            "seed": int(seed),
            "split": "calib",
            "checkpoint": str(checkpoint_path),
            "accuracy": calib_eval["accuracy"],
            "nll": calib_eval["nll"],
            "num_samples": calib_eval["num_samples"],
            "classes": IMAGENET100_SYNSETS,
        },
    )

    save_logits_npz(
        val_logits_path,
        val_eval["logits"],
        val_eval["labels"],
        metadata={
            "architecture": ARCH_NAME,
            "seed": int(seed),
            "split": "clean_val",
            "checkpoint": str(checkpoint_path),
            "accuracy": val_eval["accuracy"],
            "nll": val_eval["nll"],
            "num_samples": val_eval["num_samples"],
            "classes": IMAGENET100_SYNSETS,
        },
    )

    stage1_logit_rows.append({
        "seed": int(seed),
        "checkpoint": str(checkpoint_path),
        "calib_logits_path": str(calib_logits_path),
        "val_logits_clean_path": str(val_logits_path),
        "calib_accuracy": float(calib_eval["accuracy"]),
        "calib_nll": float(calib_eval["nll"]),
        "calib_samples": int(calib_eval["num_samples"]),
        "clean_val_accuracy": float(val_eval["accuracy"]),
        "clean_val_nll": float(val_eval["nll"]),
        "clean_val_samples": int(val_eval["num_samples"]),
    })

    print("Calib:", calib_eval["accuracy"], calib_eval["nll"], calib_logits_path)
    print("Val:", val_eval["accuracy"], val_eval["nll"], val_logits_path)

    del model
    torch.cuda.empty_cache()

stage1_logits_summary = pd.DataFrame(stage1_logit_rows)

stage1_logits_summary_path = DIRS["tables"] / "stage1_vit_clean_logits_summary.csv"
stage1_logits_summary.to_csv(stage1_logits_summary_path, index=False)

print("\nSaved Stage 1 clean logits summary:")
print(stage1_logits_summary_path)

display(stage1_logits_summary)

**Save Stage 1 completion manifest**

In [ ]:
stage1_manifest = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "architecture": ARCH_NAME,
    "num_classes": NUM_CLASSES,
    "train_config": TRAIN_CONFIG,
    "train_imagefolder_root": str(TRAIN_IMAGEFOLDER_ROOT),
    "clean_val_imagefolder_root": str(CLEAN_VAL_IMAGEFOLDER_ROOT),
    "train_calib_split_manifest": str(split_manifest_path),
    "frozen_resnet_split_source": str(FROZEN_RESNET_SPLIT_JSON),
    "split_protocol": "loaded frozen ResNet train/calib split for protocol parity with RN18 and RN50",
    "training_summary_path": str(stage1_training_summary_path),
    "clean_logits_summary_path": str(stage1_logits_summary_path),
    "checkpoint_root": str(DIRS["checkpoints"] / ARCH_NAME),
    "logits_cache_root": str(DIRS["logits_cache"] / ARCH_NAME),
    "saved_checkpoints_per_seed": [
        "best_by_acc.pt",
        "best_by_nll.pt",
        "final.pt",
    ],
    "saved_clean_logits_per_seed": [
        "calib_logits.npz",
        "val_logits_clean.npz",
    ],
}

stage1_manifest_path = DIRS["manifests"] / "stage1_vit_training_and_clean_logits_manifest.json"

with open(stage1_manifest_path, "w") as f:
    json.dump(stage1_manifest, f, indent=2)

print("Saved Stage 1 manifest:")
print(stage1_manifest_path)

print("\n========== STAGE 1 DONE ==========")
print("Checkpoints root:", DIRS["checkpoints"] / ARCH_NAME)
print("Logits cache root:", DIRS["logits_cache"] / ARCH_NAME)
print("Training summary:", stage1_training_summary_path)
print("Clean logits summary:", stage1_logits_summary_path)

**ImageNet-C dataset helper**

In [ ]:
from torchvision.datasets import ImageFolder

def make_imagenetc_condition_dataset(corruption: str, severity: int):
    """
    Builds ImageFolder dataset for one filtered ImageNet-C condition:
      LOCAL_IMAGENETC_FILTERED_ROOT / corruption / severity
    """
    condition_root = LOCAL_IMAGENETC_FILTERED_ROOT / corruption / str(severity)

    assert condition_root.exists(), f"Missing ImageNet-C condition root: {condition_root}"

    dataset = ImageFolder(
        root=str(condition_root),
        transform=eval_transform,
    )

    # Important: class order must match ImageNet-100 source class order
    assert dataset.classes == sorted(IMAGENET100_SYNSETS), (
        f"Class mismatch for {corruption}_sev{severity}\n"
        f"Dataset classes[:5]: {dataset.classes[:5]}\n"
        f"Expected[:5]: {sorted(IMAGENET100_SYNSETS)[:5]}"
    )

    return dataset


# Quick check one condition
_test_dataset = make_imagenetc_condition_dataset("brightness", 1)

print("Test condition size:", len(_test_dataset))
print("Num classes:", len(_test_dataset.classes))
print("First 10 classes:", _test_dataset.classes[:10])

del _test_dataset

**Logit caching helper**

In [ ]:
@torch.no_grad()
def cache_logits_for_dataset(model, loader, device):
    model.eval()

    all_logits = []
    all_labels = []

    total_correct = 0
    total_samples = 0
    total_nll = 0.0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = F.cross_entropy(logits, y, reduction="sum")

        pred = logits.argmax(dim=1)

        total_correct += pred.eq(y).sum().item()
        total_nll += loss.item()
        total_samples += y.size(0)

        all_logits.append(logits.detach().cpu())
        all_labels.append(y.detach().cpu())

    logits = torch.cat(all_logits, dim=0)
    labels = torch.cat(all_labels, dim=0)

    accuracy = total_correct / max(total_samples, 1)
    nll = total_nll / max(total_samples, 1)

    return {
        "logits": logits,
        "labels": labels,
        "accuracy": float(accuracy),
        "nll": float(nll),
        "num_samples": int(total_samples),
    }


def save_imagenetc_logits_pt(path, logits, labels, metadata):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    torch.save(
        {
            "logits": logits.float(),
            "labels": labels.long(),
            "metadata": metadata,
        },
        path,
    )


print("ImageNet-C logit caching helpers ready.")

**Cache ImageNet-C logits for all seeds and conditions**

In [ ]:
VIT_IMAGENETC_LOGITS_DIR = DIRS["logits_cache"] / ARCH_NAME / "imagenet_c"
VIT_IMAGENETC_LOGITS_DIR.mkdir(parents=True, exist_ok=True)

stage2_rows = []

for seed in SEEDS:
    print(f"\n{'='*90}")
    print(f"Caching ImageNet-C logits | {ARCH_NAME} | seed={seed}")
    print(f"{'='*90}")

    model, ckpt, checkpoint_path = load_vit_best_by_nll(seed)

    for corruption in CORRUPTIONS:
        for severity in SEVERITIES:
            condition = f"{corruption}_sev{severity}"

            output_path = VIT_IMAGENETC_LOGITS_DIR / f"seed{seed}_{condition}.pt"

            if output_path.exists():
                print(f"[SKIP] {output_path.name} already exists.")

                blob = torch.load(output_path, map_location="cpu", weights_only=False)
                logits = blob["logits"]
                labels = blob["labels"]

                raw_nll = F.cross_entropy(logits.float(), labels.long(), reduction="mean").item()
                raw_acc = logits.argmax(dim=1).eq(labels).float().mean().item()

                stage2_rows.append({
                    "architecture": ARCH_NAME,
                    "seed": int(seed),
                    "corruption": corruption,
                    "severity": int(severity),
                    "condition": condition,
                    "checkpoint": str(checkpoint_path),
                    "logits_path": str(output_path),
                    "num_samples": int(len(labels)),
                    "accuracy": float(raw_acc),
                    "nll": float(raw_nll),
                    "status": "skipped_existing",
                })

                continue

            dataset = make_imagenetc_condition_dataset(corruption, severity)

            loader = DataLoader(
                dataset,
                batch_size=TRAIN_CONFIG["batch_size"],
                shuffle=False,
                num_workers=TRAIN_CONFIG["num_workers"],
                pin_memory=True,
            )

            result = cache_logits_for_dataset(model, loader, DEVICE)

            metadata = {
                "architecture": ARCH_NAME,
                "seed": int(seed),
                "checkpoint": str(checkpoint_path),
                "corruption": corruption,
                "severity": int(severity),
                "condition": condition,
                "dataset_root": str(LOCAL_IMAGENETC_FILTERED_ROOT / corruption / str(severity)),
                "num_samples": result["num_samples"],
                "accuracy": result["accuracy"],
                "nll": result["nll"],
                "classes": IMAGENET100_SYNSETS,
            }

            save_imagenetc_logits_pt(
                output_path,
                result["logits"],
                result["labels"],
                metadata,
            )

            stage2_rows.append({
                "architecture": ARCH_NAME,
                "seed": int(seed),
                "corruption": corruption,
                "severity": int(severity),
                "condition": condition,
                "checkpoint": str(checkpoint_path),
                "logits_path": str(output_path),
                "num_samples": int(result["num_samples"]),
                "accuracy": float(result["accuracy"]),
                "nll": float(result["nll"]),
                "status": "computed",
            })

            print(
                f"{condition:25s} | "
                f"n={result['num_samples']:5d} | "
                f"acc={result['accuracy']:.4f} | "
                f"nll={result['nll']:.4f}"
            )

            del dataset, loader

    del model
    torch.cuda.empty_cache()

stage2_imagenetc_logits_manifest = pd.DataFrame(stage2_rows)

stage2_imagenetc_logits_manifest_path = (
    DIRS["manifests"] / "stage2_vit_imagenetc_logits_manifest.csv"
)

stage2_imagenetc_logits_manifest.to_csv(
    stage2_imagenetc_logits_manifest_path,
    index=False,
)

print("\nSaved Stage 2 ImageNet-C logits manifest:")
print(stage2_imagenetc_logits_manifest_path)

display(stage2_imagenetc_logits_manifest.head())

**Validate cached logits**

In [ ]:
expected_num_files = len(SEEDS) * len(CORRUPTIONS) * len(SEVERITIES)

cached_files = sorted(VIT_IMAGENETC_LOGITS_DIR.glob("seed*_*.pt"))

print("Expected cached files:", expected_num_files)
print("Found cached files:", len(cached_files))

assert len(cached_files) == expected_num_files, (
    f"Expected {expected_num_files} cached files, found {len(cached_files)}"
)

validation_rows = []

for seed in SEEDS:
    for corruption in CORRUPTIONS:
        for severity in SEVERITIES:
            condition = f"{corruption}_sev{severity}"
            path = VIT_IMAGENETC_LOGITS_DIR / f"seed{seed}_{condition}.pt"

            assert path.exists(), f"Missing cached logits: {path}"

            blob = torch.load(path, map_location="cpu", weights_only=False)

            logits = blob["logits"]
            labels = blob["labels"]

            assert logits.ndim == 2, f"Bad logits shape in {path}: {logits.shape}"
            assert labels.ndim == 1, f"Bad labels shape in {path}: {labels.shape}"
            assert logits.shape[0] == labels.shape[0], f"Logits/labels mismatch in {path}"
            assert logits.shape[1] == NUM_CLASSES, f"Expected {NUM_CLASSES} classes in {path}"

            acc = logits.argmax(dim=1).eq(labels).float().mean().item()
            nll = F.cross_entropy(logits.float(), labels.long(), reduction="mean").item()

            validation_rows.append({
                "architecture": ARCH_NAME,
                "seed": int(seed),
                "corruption": corruption,
                "severity": int(severity),
                "condition": condition,
                "path": str(path),
                "num_samples": int(len(labels)),
                "num_classes": int(logits.shape[1]),
                "accuracy": float(acc),
                "nll": float(nll),
            })

stage2_validation_df = pd.DataFrame(validation_rows)

stage2_validation_path = DIRS["tables"] / "stage2_vit_imagenetc_logits_validation.csv"
stage2_validation_df.to_csv(stage2_validation_path, index=False)

print("Saved Stage 2 validation table:")
print(stage2_validation_path)

display(stage2_validation_df.head())

display(
    stage2_validation_df
    .groupby(["seed"], as_index=False)
    .agg(
        n_conditions=("condition", "count"),
        total_samples=("num_samples", "sum"),
        mean_accuracy=("accuracy", "mean"),
        mean_nll=("nll", "mean"),
        min_accuracy=("accuracy", "min"),
        max_nll=("nll", "max"),
    )
    .round(6)
)

**Raw ImageNet-C summary by corruption and severity**

In [ ]:
raw_by_corruption = (
    stage2_validation_df
    .groupby(["architecture", "corruption"], as_index=False)
    .agg(
        mean_accuracy=("accuracy", "mean"),
        mean_nll=("nll", "mean"),
        n_conditions=("condition", "count"),
        total_samples=("num_samples", "sum"),
    )
    .sort_values("mean_nll", ascending=False)
)

raw_by_severity = (
    stage2_validation_df
    .groupby(["architecture", "severity"], as_index=False)
    .agg(
        mean_accuracy=("accuracy", "mean"),
        mean_nll=("nll", "mean"),
        n_conditions=("condition", "count"),
        total_samples=("num_samples", "sum"),
    )
    .sort_values("severity")
)

raw_by_corruption_path = DIRS["tables"] / "stage2_vit_raw_imagenetc_by_corruption.csv"
raw_by_severity_path = DIRS["tables"] / "stage2_vit_raw_imagenetc_by_severity.csv"

raw_by_corruption.to_csv(raw_by_corruption_path, index=False)
raw_by_severity.to_csv(raw_by_severity_path, index=False)

print("Saved raw by-corruption summary:")
print(raw_by_corruption_path)

print("\nSaved raw by-severity summary:")
print(raw_by_severity_path)

print("\nRaw ImageNet-C by corruption:")
display(raw_by_corruption.round(6))

print("\nRaw ImageNet-C by severity:")
display(raw_by_severity.round(6))

**Save Stage 2 completion manifest**

In [ ]:
stage2_manifest = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "architecture": ARCH_NAME,
    "checkpoint_source": "best_by_nll.pt",
    "imagenetc_filtered_root": str(LOCAL_IMAGENETC_FILTERED_ROOT),
    "imagenetc_logits_dir": str(VIT_IMAGENETC_LOGITS_DIR),
    "num_seeds": len(SEEDS),
    "seeds": SEEDS,
    "corruptions": CORRUPTIONS,
    "severities": SEVERITIES,
    "expected_num_cached_files": expected_num_files,
    "actual_num_cached_files": len(cached_files),
    "logits_manifest_path": str(stage2_imagenetc_logits_manifest_path),
    "validation_table_path": str(stage2_validation_path),
    "raw_by_corruption_path": str(raw_by_corruption_path),
    "raw_by_severity_path": str(raw_by_severity_path),
    "notes": [
        "ImageNet-C was filtered to the ImageNet-100 class set.",
        "Logits were cached for each seed, corruption, and severity.",
        "These logits will be used for source TS evaluation, oracle diagnostics, stream entropy, V2/V3, and LOCO/LOSO downstream metrics.",
    ],
}

stage2_manifest_path = DIRS["manifests"] / "stage2_vit_imagenetc_logits_manifest.json"

with open(stage2_manifest_path, "w") as f:
    json.dump(stage2_manifest, f, indent=2)

print("Saved Stage 2 completion manifest:")
print(stage2_manifest_path)

print("\n========== STAGE 2 DONE ==========")
print("Cached logits dir:", VIT_IMAGENETC_LOGITS_DIR)
print("Cached files:", len(cached_files))
print("Validation table:", stage2_validation_path)
print("Raw by corruption:", raw_by_corruption_path)
print("Raw by severity:", raw_by_severity_path)

**Temperature scaling and calibration metric helpers**

In [ ]:
def load_npz_logits_labels(path: Path):
    path = Path(path)
    assert path.exists(), f"Missing logits file: {path}"

    blob = np.load(path, allow_pickle=True)
    logits = torch.tensor(blob["logits"], dtype=torch.float32)
    labels = torch.tensor(blob["labels"], dtype=torch.long)

    assert logits.ndim == 2, f"Bad logits shape: {logits.shape}"
    assert labels.ndim == 1, f"Bad labels shape: {labels.shape}"
    assert logits.shape[0] == labels.shape[0], "Logits/labels length mismatch"
    assert logits.shape[1] == NUM_CLASSES, f"Expected {NUM_CLASSES} classes, got {logits.shape[1]}"

    return logits, labels


def load_pt_logits_labels(path: Path):
    path = Path(path)
    assert path.exists(), f"Missing logits file: {path}"

    blob = torch.load(path, map_location="cpu", weights_only=False)

    logits = blob["logits"].detach().cpu().float()
    labels = blob["labels"].detach().cpu().long()

    assert logits.ndim == 2, f"Bad logits shape: {logits.shape}"
    assert labels.ndim == 1, f"Bad labels shape: {labels.shape}"
    assert logits.shape[0] == labels.shape[0], "Logits/labels length mismatch"
    assert logits.shape[1] == NUM_CLASSES, f"Expected {NUM_CLASSES} classes, got {logits.shape[1]}"

    return logits, labels


def apply_temperature(logits: torch.Tensor, temperature: float):
    assert temperature > 0, f"Temperature must be positive, got {temperature}"
    return logits.detach().cpu().float() / float(temperature)


def fit_temperature_nll(
    logits: torch.Tensor,
    labels: torch.Tensor,
    init_temperature: float = 1.0,
    max_iter: int = 100,
    lr: float = 0.05,
    min_temperature: float = 0.05,
    max_temperature: float = 10.0,
):
    logits = logits.detach().cpu().float()
    labels = labels.detach().cpu().long()

    log_temperature = torch.tensor(
        [math.log(init_temperature)],
        dtype=torch.float32,
        requires_grad=True,
    )

    optimizer = torch.optim.LBFGS(
        [log_temperature],
        lr=lr,
        max_iter=max_iter,
        line_search_fn="strong_wolfe",
    )

    def closure():
        optimizer.zero_grad()

        temperature = torch.exp(log_temperature)
        temperature = torch.clamp(
            temperature,
            min=min_temperature,
            max=max_temperature,
        )

        loss = F.cross_entropy(logits / temperature, labels, reduction="mean")
        loss.backward()
        return loss

    optimizer.step(closure)

    with torch.no_grad():
        temperature = torch.exp(log_temperature).item()
        temperature = float(np.clip(temperature, min_temperature, max_temperature))

        nll_before = F.cross_entropy(logits, labels, reduction="mean").item()
        nll_after = F.cross_entropy(logits / temperature, labels, reduction="mean").item()

    return {
        "temperature": float(temperature),
        "log_temperature": float(math.log(temperature)),
        "nll_before": float(nll_before),
        "nll_after": float(nll_after),
        "nll_improvement": float(nll_before - nll_after),
    }


print("Temperature scaling helpers ready.")

**Metric functions**

In [ ]:
def nll_from_logits(logits: torch.Tensor, labels: torch.Tensor) -> float:
    return F.cross_entropy(logits.float(), labels.long(), reduction="mean").item()


def brier_from_logits(logits: torch.Tensor, labels: torch.Tensor) -> float:
    probs = F.softmax(logits.float(), dim=1)
    one_hot = F.one_hot(labels.long(), num_classes=probs.shape[1]).float()
    return ((probs - one_hot) ** 2).sum(dim=1).mean().item()


def ece_from_logits(logits: torch.Tensor, labels: torch.Tensor, n_bins: int = 15) -> float:
    probs = F.softmax(logits.float(), dim=1)
    confidences, predictions = probs.max(dim=1)
    accuracies = predictions.eq(labels.long()).float()

    bin_edges = torch.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        lo = bin_edges[i]
        hi = bin_edges[i + 1]

        if i == 0:
            in_bin = (confidences >= lo) & (confidences <= hi)
        else:
            in_bin = (confidences > lo) & (confidences <= hi)

        prop = in_bin.float().mean().item()

        if prop > 0:
            acc_bin = accuracies[in_bin].mean().item()
            conf_bin = confidences[in_bin].mean().item()
            ece += abs(acc_bin - conf_bin) * prop

    return float(ece)


def adaptive_ece_from_logits(logits: torch.Tensor, labels: torch.Tensor, n_bins: int = 15) -> float:
    probs = F.softmax(logits.float(), dim=1)
    confidences, predictions = probs.max(dim=1)
    accuracies = predictions.eq(labels.long()).float()

    confidences = confidences.detach().cpu()
    accuracies = accuracies.detach().cpu()

    n = len(confidences)
    if n == 0:
        return 0.0

    order = torch.argsort(confidences)
    confidences = confidences[order]
    accuracies = accuracies[order]

    bin_sizes = [n // n_bins] * n_bins
    for i in range(n % n_bins):
        bin_sizes[i] += 1

    aece = 0.0
    start = 0

    for size in bin_sizes:
        if size == 0:
            continue

        end = start + size
        conf_bin = confidences[start:end]
        acc_bin = accuracies[start:end]

        aece += abs(conf_bin.mean().item() - acc_bin.mean().item()) * (size / n)
        start = end

    return float(aece)


def metrics_from_logits(logits: torch.Tensor, labels: torch.Tensor) -> dict:
    logits = logits.detach().cpu().float()
    labels = labels.detach().cpu().long()

    probs = F.softmax(logits, dim=1)
    confidence, prediction = probs.max(dim=1)

    accuracy = prediction.eq(labels).float().mean().item()
    mean_confidence = confidence.mean().item()
    signed_gap = mean_confidence - accuracy

    out = {
        "accuracy": float(accuracy),
        "mean_confidence": float(mean_confidence),
        "signed_gap": float(signed_gap),
        "abs_signed_gap": float(abs(signed_gap)),
        "nll": float(nll_from_logits(logits, labels)),
        "brier": float(brier_from_logits(logits, labels)),
        "ece15": float(ece_from_logits(logits, labels, n_bins=15)),
        "aece15": float(adaptive_ece_from_logits(logits, labels, n_bins=15)),
    }

    assert 0.0 <= out["ece15"] <= 1.0, f"Invalid ECE15: {out['ece15']}"
    assert 0.0 <= out["aece15"] <= 1.0, f"Invalid AECE15: {out['aece15']}"

    return out


METRICS = [
    "accuracy",
    "mean_confidence",
    "signed_gap",
    "abs_signed_gap",
    "nll",
    "brier",
    "ece15",
    "aece15",
]

print("Metric functions ready.")

**Refit source TS from ViT calibration logits**

In [ ]:
source_ts_rows = []

SOURCE_T_BY_SEED = {}

for seed in SEEDS:
    calib_logits_path = DIRS["logits_cache"] / ARCH_NAME / f"seed_{seed}" / "calib_logits.npz"

    print(f"\nRefitting source TS | {ARCH_NAME} | seed={seed}")
    print("Calib logits:", calib_logits_path)

    logits, labels = load_npz_logits_labels(calib_logits_path)

    fit = fit_temperature_nll(
        logits=logits,
        labels=labels,
        init_temperature=1.0,
        max_iter=100,
        lr=0.05,
        min_temperature=0.05,
        max_temperature=10.0,
    )

    SOURCE_T_BY_SEED[int(seed)] = float(fit["temperature"])

    source_ts_rows.append({
        "architecture": ARCH_NAME,
        "information_regime": "source_only",
        "method": "source_global_ts",
        "fit_data": "clean_source_calibration_logits",
        "seed": int(seed),
        "calib_logits_path": str(calib_logits_path),
        "n_samples": int(len(labels)),
        "n_classes": int(logits.shape[1]),
        "temperature": float(fit["temperature"]),
        "log_temperature": float(fit["log_temperature"]),
        "nll_before_ts": float(fit["nll_before"]),
        "nll_after_ts": float(fit["nll_after"]),
        "nll_improvement": float(fit["nll_improvement"]),
    })

source_ts_refit_df = pd.DataFrame(source_ts_rows)

assert len(source_ts_refit_df) == len(SEEDS)
assert source_ts_refit_df["temperature"].gt(0).all()
assert (
    source_ts_refit_df["nll_after_ts"]
    <= source_ts_refit_df["nll_before_ts"] + 1e-6
).all(), "Some source TS fits worsened calibration NLL."

source_ts_refit_path = DIRS["source_ts"] / "stage3_vit_refit_source_ts_from_calib_logits.csv"
source_ts_refit_df.to_csv(source_ts_refit_path, index=False)

print("\nSaved source TS refit table:")
print(source_ts_refit_path)

print("\nSource temperatures:")
for seed, temp in SOURCE_T_BY_SEED.items():
    print(f"seed {seed}: T = {temp:.6f}")

display(source_ts_refit_df.round(6))

**Evaluate raw and source TS on clean validation logits**

In [ ]:
clean_val_rows = []

for seed in SEEDS:
    val_logits_path = DIRS["logits_cache"] / ARCH_NAME / f"seed_{seed}" / "val_logits_clean.npz"
    source_T = SOURCE_T_BY_SEED[int(seed)]

    logits, labels = load_npz_logits_labels(val_logits_path)

    method_to_temperature = {
        "raw": 1.0,
        "source_global_ts": source_T,
    }

    for method, temperature in method_to_temperature.items():
        scaled_logits = apply_temperature(logits, temperature)
        met = metrics_from_logits(scaled_logits, labels)

        for metric_name, value in met.items():
            clean_val_rows.append({
                "architecture": ARCH_NAME,
                "split_name": "clean_val",
                "information_regime": "raw" if method == "raw" else "source_only",
                "seed": int(seed),
                "method": method,
                "temperature": float(temperature),
                "metric": metric_name,
                "value": float(value),
                "logits_path": str(val_logits_path),
            })

clean_val_eval_long = pd.DataFrame(clean_val_rows)

clean_val_eval_path = DIRS["tables"] / "stage3_vit_clean_val_raw_source_ts_eval_long.csv"
clean_val_eval_long.to_csv(clean_val_eval_path, index=False)

print("Saved clean val raw/source TS eval long:")
print(clean_val_eval_path)

display(clean_val_eval_long.head())

**Clean validation condition-wide and summary**

In [ ]:
clean_val_wide = (
    clean_val_eval_long
    .pivot_table(
        index=[
            "architecture",
            "split_name",
            "information_regime",
            "seed",
            "method",
            "temperature",
            "logits_path",
        ],
        columns="metric",
        values="value",
        aggfunc="mean",
    )
    .reset_index()
)

clean_val_wide.columns.name = None

missing_metrics = [m for m in METRICS if m not in clean_val_wide.columns]
assert len(missing_metrics) == 0, f"Missing metrics: {missing_metrics}"

assert clean_val_wide["ece15"].between(0, 1).all()
assert clean_val_wide["aece15"].between(0, 1).all()

clean_val_wide_path = DIRS["tables"] / "stage3_vit_clean_val_raw_source_ts_wide.csv"
clean_val_wide.to_csv(clean_val_wide_path, index=False)

clean_val_summary = (
    clean_val_wide
    .groupby(["architecture", "split_name", "information_regime", "method"], as_index=False)
    .agg(
        accuracy_mean=("accuracy", "mean"),
        mean_confidence_mean=("mean_confidence", "mean"),
        signed_gap_mean=("signed_gap", "mean"),
        abs_signed_gap_mean=("abs_signed_gap", "mean"),
        nll_mean=("nll", "mean"),
        brier_mean=("brier", "mean"),
        ece15_mean=("ece15", "mean"),
        aece15_mean=("aece15", "mean"),
        temperature_mean=("temperature", "mean"),
        temperature_min=("temperature", "min"),
        temperature_max=("temperature", "max"),
        n_seeds=("seed", "count"),
    )
)

clean_val_summary_path = DIRS["tables"] / "stage3_vit_clean_val_raw_source_ts_summary.csv"
clean_val_summary.to_csv(clean_val_summary_path, index=False)

print("Saved clean val wide:")
print(clean_val_wide_path)

print("\nSaved clean val summary:")
print(clean_val_summary_path)

display(clean_val_summary.round(6))

**Evaluate raw and source TS on ImageNet-C logits**

In [ ]:
VIT_IMAGENETC_LOGITS_DIR = DIRS["logits_cache"] / ARCH_NAME / "imagenet_c"
VIT_IMAGENETC_LOGITS_DIR.mkdir(parents=True, exist_ok=True)

# Sanity check: confirm the 150 cached files are there from your previous session
expected = len(SEEDS) * len(CORRUPTIONS) * len(SEVERITIES)
cached_files = sorted(VIT_IMAGENETC_LOGITS_DIR.glob("seed*_*.pt"))
print(f"Found {len(cached_files)} / {expected} cached ImageNet-C logit files on Drive")
assert len(cached_files) == expected, "Some cached ImageNet-C logits missing — check Drive"
print("OK — ready to resume Stage 3")

In [ ]:
stage3_imagenetc_rows = []

for seed in SEEDS:
    source_T = SOURCE_T_BY_SEED[int(seed)]

    print(f"\nEvaluating ImageNet-C raw/source TS | seed={seed} | source_T={source_T:.6f}")

    for corruption in CORRUPTIONS:
        for severity in SEVERITIES:
            condition = f"{corruption}_sev{severity}"
            logits_path = VIT_IMAGENETC_LOGITS_DIR / f"seed{seed}_{condition}.pt"

            logits, labels = load_pt_logits_labels(logits_path)

            method_to_temperature = {
                "raw": 1.0,
                "source_global_ts": source_T,
            }

            for method, temperature in method_to_temperature.items():
                scaled_logits = apply_temperature(logits, temperature)
                met = metrics_from_logits(scaled_logits, labels)

                for metric_name, value in met.items():
                    stage3_imagenetc_rows.append({
                        "architecture": ARCH_NAME,
                        "split_name": "imagenet_c_filtered",
                        "heldout_type": "none",
                        "heldout_value": "none",
                        "information_regime": "raw" if method == "raw" else "source_only",
                        "seed": int(seed),
                        "method": method,
                        "corruption": corruption,
                        "severity": int(severity),
                        "condition": condition,
                        "temperature": float(temperature),
                        "metric": metric_name,
                        "value": float(value),
                        "logits_path": str(logits_path),
                    })

stage3_imagenetc_eval_long = pd.DataFrame(stage3_imagenetc_rows)

expected_rows = len(SEEDS) * len(CORRUPTIONS) * len(SEVERITIES) * 2 * len(METRICS)
assert len(stage3_imagenetc_eval_long) == expected_rows, (
    f"Expected {expected_rows} rows, got {len(stage3_imagenetc_eval_long)}"
)

assert stage3_imagenetc_eval_long["value"].notna().all()

stage3_imagenetc_eval_path = DIRS["tables"] / "stage3_vit_imagenetc_raw_source_ts_eval_long.csv"
stage3_imagenetc_eval_long.to_csv(stage3_imagenetc_eval_path, index=False)

print("\nSaved ImageNet-C raw/source TS eval long:")
print(stage3_imagenetc_eval_path)

display(stage3_imagenetc_eval_long.head())

**ImageNet-C condition-wide table**

In [ ]:
stage3_imagenetc_wide = (
    stage3_imagenetc_eval_long
    .pivot_table(
        index=[
            "architecture",
            "split_name",
            "heldout_type",
            "heldout_value",
            "information_regime",
            "seed",
            "method",
            "corruption",
            "severity",
            "condition",
            "temperature",
            "logits_path",
        ],
        columns="metric",
        values="value",
        aggfunc="mean",
    )
    .reset_index()
)

stage3_imagenetc_wide.columns.name = None

missing_metrics = [m for m in METRICS if m not in stage3_imagenetc_wide.columns]
assert len(missing_metrics) == 0, f"Missing metrics: {missing_metrics}"

assert stage3_imagenetc_wide["ece15"].between(0, 1).all(), "ECE15 outside [0,1]"
assert stage3_imagenetc_wide["aece15"].between(0, 1).all(), "AECE15 outside [0,1]"

expected_wide_rows = len(SEEDS) * len(CORRUPTIONS) * len(SEVERITIES) * 2
assert len(stage3_imagenetc_wide) == expected_wide_rows, (
    f"Expected {expected_wide_rows} wide rows, got {len(stage3_imagenetc_wide)}"
)

stage3_imagenetc_wide_path = DIRS["tables"] / "stage3_vit_imagenetc_raw_source_ts_condition_wide.csv"
stage3_imagenetc_wide.to_csv(stage3_imagenetc_wide_path, index=False)

print("Saved ImageNet-C raw/source TS condition-wide table:")
print(stage3_imagenetc_wide_path)

display(stage3_imagenetc_wide.head())

**Source TS summary on ImageNet-C**

In [ ]:
stage3_source_reference_summary = (
    stage3_imagenetc_wide
    .groupby(
        [
            "architecture",
            "split_name",
            "heldout_type",
            "heldout_value",
            "information_regime",
            "method",
        ],
        as_index=False,
    )
    .agg(
        accuracy_mean=("accuracy", "mean"),
        mean_confidence_mean=("mean_confidence", "mean"),
        signed_gap_mean=("signed_gap", "mean"),
        abs_signed_gap_mean=("abs_signed_gap", "mean"),
        nll_mean=("nll", "mean"),
        brier_mean=("brier", "mean"),
        ece15_mean=("ece15", "mean"),
        aece15_mean=("aece15", "mean"),
        temperature_mean=("temperature", "mean"),
        temperature_min=("temperature", "min"),
        temperature_max=("temperature", "max"),
        n_conditions=("nll", "count"),
    )
)

assert stage3_source_reference_summary["ece15_mean"].between(0, 1).all()
assert stage3_source_reference_summary["aece15_mean"].between(0, 1).all()
assert stage3_source_reference_summary["ece15_mean"].max() < 1.0, (
    "ECE summary is suspicious. ECE should be in [0,1], not a count."
)

stage3_source_reference_summary_path = DIRS["tables"] / "stage3_vit_source_reference_summary.csv"
stage3_source_reference_summary.to_csv(stage3_source_reference_summary_path, index=False)

print("Saved source reference summary:")
print(stage3_source_reference_summary_path)

display(stage3_source_reference_summary.round(6))

**Source TS deltas versus raw**

In [ ]:
raw_wide = stage3_imagenetc_wide[
    stage3_imagenetc_wide["method"] == "raw"
].copy()

source_wide = stage3_imagenetc_wide[
    stage3_imagenetc_wide["method"] == "source_global_ts"
].copy()

merge_keys = [
    "architecture",
    "seed",
    "corruption",
    "severity",
    "condition",
]

raw_keep = merge_keys + METRICS
source_keep = merge_keys + METRICS + ["temperature"]

raw_wide = raw_wide[raw_keep].rename(columns={m: f"raw_{m}" for m in METRICS})
source_wide = source_wide[source_keep].rename(
    columns={m: f"source_ts_{m}" for m in METRICS}
).rename(columns={"temperature": "source_ts_temperature"})

stage3_source_delta_wide = source_wide.merge(
    raw_wide,
    on=merge_keys,
    how="inner",
)

expected_delta_rows = len(SEEDS) * len(CORRUPTIONS) * len(SEVERITIES)
assert len(stage3_source_delta_wide) == expected_delta_rows, (
    f"Expected {expected_delta_rows}, got {len(stage3_source_delta_wide)}"
)

for metric in METRICS:
    stage3_source_delta_wide[f"delta_{metric}_source_ts_minus_raw"] = (
        stage3_source_delta_wide[f"source_ts_{metric}"] -
        stage3_source_delta_wide[f"raw_{metric}"]
    )

assert (stage3_source_delta_wide["delta_accuracy_source_ts_minus_raw"].abs() < 1e-10).all(), (
    "Source TS changed accuracy. Positive scalar temperature should preserve argmax."
)

assert stage3_source_delta_wide["source_ts_ece15"].between(0, 1).all()
assert stage3_source_delta_wide["raw_ece15"].between(0, 1).all()

stage3_source_delta_wide_path = DIRS["tables"] / "stage3_vit_source_ts_delta_vs_raw_wide.csv"
stage3_source_delta_wide.to_csv(stage3_source_delta_wide_path, index=False)

stage3_source_delta_summary = (
    stage3_source_delta_wide
    .groupby(["architecture"], as_index=False)
    .agg(
        delta_nll_mean=("delta_nll_source_ts_minus_raw", "mean"),
        delta_brier_mean=("delta_brier_source_ts_minus_raw", "mean"),
        delta_ece15_mean=("delta_ece15_source_ts_minus_raw", "mean"),
        delta_aece15_mean=("delta_aece15_source_ts_minus_raw", "mean"),
        delta_signed_gap_mean=("delta_signed_gap_source_ts_minus_raw", "mean"),
        delta_abs_signed_gap_mean=("delta_abs_signed_gap_source_ts_minus_raw", "mean"),
        source_ts_temperature_mean=("source_ts_temperature", "mean"),
        source_ts_temperature_min=("source_ts_temperature", "min"),
        source_ts_temperature_max=("source_ts_temperature", "max"),
        n_conditions=("delta_nll_source_ts_minus_raw", "count"),
    )
)

stage3_source_delta_summary_path = DIRS["tables"] / "stage3_vit_source_ts_delta_vs_raw_summary.csv"
stage3_source_delta_summary.to_csv(stage3_source_delta_summary_path, index=False)

print("Saved source TS delta wide:")
print(stage3_source_delta_wide_path)

print("\nSaved source TS delta summary:")
print(stage3_source_delta_summary_path)

display(stage3_source_delta_summary.round(6))

**Summaries by corruption and severity**

In [ ]:
stage3_by_corruption = (
    stage3_imagenetc_wide
    .groupby(["architecture", "method", "corruption"], as_index=False)
    .agg(
        accuracy_mean=("accuracy", "mean"),
        nll_mean=("nll", "mean"),
        brier_mean=("brier", "mean"),
        ece15_mean=("ece15", "mean"),
        aece15_mean=("aece15", "mean"),
        signed_gap_mean=("signed_gap", "mean"),
        abs_signed_gap_mean=("abs_signed_gap", "mean"),
        temperature_mean=("temperature", "mean"),
        n_conditions=("condition", "count"),
    )
)

stage3_by_severity = (
    stage3_imagenetc_wide
    .groupby(["architecture", "method", "severity"], as_index=False)
    .agg(
        accuracy_mean=("accuracy", "mean"),
        nll_mean=("nll", "mean"),
        brier_mean=("brier", "mean"),
        ece15_mean=("ece15", "mean"),
        aece15_mean=("aece15", "mean"),
        signed_gap_mean=("signed_gap", "mean"),
        abs_signed_gap_mean=("abs_signed_gap", "mean"),
        temperature_mean=("temperature", "mean"),
        n_conditions=("condition", "count"),
    )
)

stage3_by_corruption_path = DIRS["tables"] / "stage3_vit_raw_source_ts_by_corruption.csv"
stage3_by_severity_path = DIRS["tables"] / "stage3_vit_raw_source_ts_by_severity.csv"

stage3_by_corruption.to_csv(stage3_by_corruption_path, index=False)
stage3_by_severity.to_csv(stage3_by_severity_path, index=False)

print("Saved by-corruption summary:")
print(stage3_by_corruption_path)

print("\nSaved by-severity summary:")
print(stage3_by_severity_path)

print("\nBy severity:")
display(stage3_by_severity.round(6))

print("\nBy corruption:")
display(stage3_by_corruption.round(6).head(30))

**Save Stage 3 completion manifest**

In [ ]:
stage3_manifest = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "architecture": ARCH_NAME,
    "stage": "stage3_refit_source_ts_and_evaluate_raw_source_ts",
    "source_ts_refit_path": str(source_ts_refit_path),
    "clean_val_eval_long_path": str(clean_val_eval_path),
    "clean_val_wide_path": str(clean_val_wide_path),
    "clean_val_summary_path": str(clean_val_summary_path),
    "imagenetc_eval_long_path": str(stage3_imagenetc_eval_path),
    "imagenetc_condition_wide_path": str(stage3_imagenetc_wide_path),
    "source_reference_summary_path": str(stage3_source_reference_summary_path),
    "source_ts_delta_wide_path": str(stage3_source_delta_wide_path),
    "source_ts_delta_summary_path": str(stage3_source_delta_summary_path),
    "by_corruption_path": str(stage3_by_corruption_path),
    "by_severity_path": str(stage3_by_severity_path),
    "checks": {
        "source_ts_refit_from_clean_calib_logits": True,
        "imagenetc_eval_uses_cached_logits": True,
        "ece15_computed_from_logits": True,
        "aece15_computed_from_logits": True,
        "ece15_values_between_0_and_1": bool(stage3_imagenetc_wide["ece15"].between(0, 1).all()),
        "aece15_values_between_0_and_1": bool(stage3_imagenetc_wide["aece15"].between(0, 1).all()),
        "source_ts_preserves_accuracy": bool(
            (stage3_source_delta_wide["delta_accuracy_source_ts_minus_raw"].abs() < 1e-10).all()
        ),
    },
    "source_temperatures": {str(k): float(v) for k, v in SOURCE_T_BY_SEED.items()},
}

stage3_manifest_path = DIRS["manifests"] / "stage3_vit_source_ts_and_baseline_eval_manifest.json"

with open(stage3_manifest_path, "w") as f:
    json.dump(stage3_manifest, f, indent=2)

print("Saved Stage 3 manifest:")
print(stage3_manifest_path)

print("\n========== STAGE 3 DONE ==========")
print("Source TS refit:", source_ts_refit_path)
print("ImageNet-C condition-wide:", stage3_imagenetc_wide_path)
print("Source reference summary:", stage3_source_reference_summary_path)
print("Source TS delta summary:", stage3_source_delta_summary_path)

display(stage3_source_reference_summary.round(6))
display(stage3_source_delta_summary.round(6))

**Stage 4.1 — Target-oracle fitting helper**

In [ ]:
def fit_target_oracle_for_condition(seed: int, corruption: str, severity: int):
    """
    Fits oracle temperature on one target ImageNet-C condition.
    This uses target labels, so it is diagnostic only.
    """
    condition = f"{corruption}_sev{severity}"
    logits_path = VIT_IMAGENETC_LOGITS_DIR / f"seed{seed}_{condition}.pt"

    logits, labels = load_pt_logits_labels(logits_path)

    source_T = SOURCE_T_BY_SEED[int(seed)]

    fit = fit_temperature_nll(
        logits=logits,
        labels=labels,
        init_temperature=source_T,
        max_iter=100,
        lr=0.05,
        min_temperature=0.05,
        max_temperature=10.0,
    )

    return {
        "architecture": ARCH_NAME,
        "seed": int(seed),
        "corruption": corruption,
        "severity": int(severity),
        "condition": condition,
        "logits_path": str(logits_path),
        "source_T": float(source_T),
        "oracle_T": float(fit["temperature"]),
        "oracle_log_T": float(fit["log_temperature"]),
        "nll_before_oracle": float(fit["nll_before"]),
        "nll_after_oracle": float(fit["nll_after"]),
        "oracle_nll_improvement": float(fit["nll_improvement"]),
        "num_samples": int(len(labels)),
    }


print("Target-oracle fitting helper ready.")

**Fit target-oracle TS for every seed/condition**

In [ ]:
target_oracle_rows = []

for seed in SEEDS:
    print(f"\n{'='*90}")
    print(f"Fitting target-oracle temperatures | {ARCH_NAME} | seed={seed}")
    print(f"{'='*90}")

    for corruption in CORRUPTIONS:
        for severity in SEVERITIES:
            row = fit_target_oracle_for_condition(
                seed=seed,
                corruption=corruption,
                severity=severity,
            )

            target_oracle_rows.append(row)

            print(
                f"{row['condition']:25s} | "
                f"source_T={row['source_T']:.4f} | "
                f"oracle_T={row['oracle_T']:.4f} | "
                f"ΔNLL_fit={row['oracle_nll_improvement']:.4f}"
            )

target_oracle_temperature_df = pd.DataFrame(target_oracle_rows)

expected_oracle_rows = len(SEEDS) * len(CORRUPTIONS) * len(SEVERITIES)

assert len(target_oracle_temperature_df) == expected_oracle_rows, (
    f"Expected {expected_oracle_rows}, got {len(target_oracle_temperature_df)}"
)

assert target_oracle_temperature_df["oracle_T"].gt(0).all()
assert target_oracle_temperature_df["nll_after_oracle"].notna().all()

bad_oracle_fits = target_oracle_temperature_df[
    target_oracle_temperature_df["nll_after_oracle"]
    > target_oracle_temperature_df["nll_before_oracle"] + 1e-6
]

assert len(bad_oracle_fits) == 0, (
    "Some target-oracle fits worsened NLL:\n"
    + bad_oracle_fits.head().to_string(index=False)
)

target_oracle_temperature_path = (
    DIRS["target_oracle_diagnostics"] / "stage4_vit_target_oracle_temperature_by_condition.csv"
)

target_oracle_temperature_df.to_csv(target_oracle_temperature_path, index=False)

print("\nSaved target-oracle temperature table:")
print(target_oracle_temperature_path)

display(target_oracle_temperature_df.head())

**Evaluate target-oracle TS downstream**

In [ ]:
target_oracle_eval_rows = []

for _, row in target_oracle_temperature_df.iterrows():
    seed = int(row["seed"])
    corruption = row["corruption"]
    severity = int(row["severity"])
    condition = row["condition"]
    oracle_T = float(row["oracle_T"])

    logits_path = VIT_IMAGENETC_LOGITS_DIR / f"seed{seed}_{condition}.pt"
    logits, labels = load_pt_logits_labels(logits_path)

    scaled_logits = apply_temperature(logits, oracle_T)
    met = metrics_from_logits(scaled_logits, labels)

    for metric_name, value in met.items():
        target_oracle_eval_rows.append({
            "architecture": ARCH_NAME,
            "split_name": "imagenet_c_filtered",
            "heldout_type": "none",
            "heldout_value": "none",
            "information_regime": "target_oracle_diagnostic_only",
            "seed": int(seed),
            "method": "target_oracle_ts_condition",
            "corruption": corruption,
            "severity": severity,
            "condition": condition,
            "temperature": oracle_T,
            "metric": metric_name,
            "value": float(value),
            "logits_path": str(logits_path),
        })

target_oracle_eval_long = pd.DataFrame(target_oracle_eval_rows)

expected_eval_rows = expected_oracle_rows * len(METRICS)

assert len(target_oracle_eval_long) == expected_eval_rows, (
    f"Expected {expected_eval_rows}, got {len(target_oracle_eval_long)}"
)

assert target_oracle_eval_long["value"].notna().all()

target_oracle_eval_path = (
    DIRS["target_oracle_diagnostics"] / "stage4_vit_target_oracle_eval_long.csv"
)

target_oracle_eval_long.to_csv(target_oracle_eval_path, index=False)

print("Saved target-oracle eval long:")
print(target_oracle_eval_path)

display(target_oracle_eval_long.head())

**Target-oracle condition-wide table**

In [ ]:
target_oracle_condition_wide = (
    target_oracle_eval_long
    .pivot_table(
        index=[
            "architecture",
            "split_name",
            "heldout_type",
            "heldout_value",
            "information_regime",
            "seed",
            "method",
            "corruption",
            "severity",
            "condition",
            "temperature",
            "logits_path",
        ],
        columns="metric",
        values="value",
        aggfunc="mean",
    )
    .reset_index()
)

target_oracle_condition_wide.columns.name = None

missing_metrics = [m for m in METRICS if m not in target_oracle_condition_wide.columns]
assert len(missing_metrics) == 0, f"Missing metrics: {missing_metrics}"

assert target_oracle_condition_wide["ece15"].between(0, 1).all()
assert target_oracle_condition_wide["aece15"].between(0, 1).all()

expected_wide_rows = len(SEEDS) * len(CORRUPTIONS) * len(SEVERITIES)
assert len(target_oracle_condition_wide) == expected_wide_rows, (
    f"Expected {expected_wide_rows}, got {len(target_oracle_condition_wide)}"
)

target_oracle_condition_wide_path = (
    DIRS["target_oracle_diagnostics"] / "stage4_vit_target_oracle_condition_wide.csv"
)

target_oracle_condition_wide.to_csv(target_oracle_condition_wide_path, index=False)

print("Saved target-oracle condition-wide table:")
print(target_oracle_condition_wide_path)

display(target_oracle_condition_wide.head())

**Add deltas versus corrected source TS**

In [ ]:
source_ts_wide = stage3_imagenetc_wide[
    stage3_imagenetc_wide["method"] == "source_global_ts"
].copy()

source_keep_cols = [
    "architecture",
    "seed",
    "corruption",
    "severity",
    "condition",
] + METRICS

source_ts_baseline = source_ts_wide[source_keep_cols].rename(
    columns={m: f"source_ts_{m}" for m in METRICS}
)

merge_keys = [
    "architecture",
    "seed",
    "corruption",
    "severity",
    "condition",
]

target_oracle_delta_wide = target_oracle_condition_wide.merge(
    source_ts_baseline,
    on=merge_keys,
    how="left",
)

assert len(target_oracle_delta_wide) == len(target_oracle_condition_wide), (
    "Merge with source TS changed row count."
)

assert target_oracle_delta_wide["source_ts_nll"].notna().all(), (
    "Missing source TS baseline after merge."
)

for metric in METRICS:
    target_oracle_delta_wide[f"delta_{metric}_vs_source_ts"] = (
        target_oracle_delta_wide[metric] -
        target_oracle_delta_wide[f"source_ts_{metric}"]
    )

assert (target_oracle_delta_wide["delta_accuracy_vs_source_ts"].abs() < 1e-10).all(), (
    "Temperature scaling changed accuracy. This should not happen."
)

# Add oracle/source temperature metadata
oracle_meta = target_oracle_temperature_df[
    [
        "architecture",
        "seed",
        "corruption",
        "severity",
        "condition",
        "source_T",
        "oracle_T",
        "oracle_log_T",
    ]
].copy()

target_oracle_delta_wide = target_oracle_delta_wide.merge(
    oracle_meta,
    on=merge_keys,
    how="left",
)

assert target_oracle_delta_wide["oracle_T"].notna().all()

assert np.allclose(
    target_oracle_delta_wide["temperature"].to_numpy(dtype=float),
    target_oracle_delta_wide["oracle_T"].to_numpy(dtype=float),
    atol=1e-10,
), "Evaluated temperature does not match oracle_T."

target_oracle_delta_wide_path = (
    DIRS["target_oracle_diagnostics"] / "stage4_vit_target_oracle_delta_vs_source_ts_wide.csv"
)

target_oracle_delta_wide.to_csv(target_oracle_delta_wide_path, index=False)

print("Saved target-oracle delta wide:")
print(target_oracle_delta_wide_path)

display(target_oracle_delta_wide.head())

**Target-oracle global summary**

In [ ]:
target_oracle_delta_summary = (
    target_oracle_delta_wide
    .groupby(
        [
            "architecture",
            "split_name",
            "information_regime",
            "method",
        ],
        as_index=False,
    )
    .agg(
        delta_nll_mean=("delta_nll_vs_source_ts", "mean"),
        delta_brier_mean=("delta_brier_vs_source_ts", "mean"),
        delta_ece15_mean=("delta_ece15_vs_source_ts", "mean"),
        delta_aece15_mean=("delta_aece15_vs_source_ts", "mean"),
        delta_signed_gap_mean=("delta_signed_gap_vs_source_ts", "mean"),
        delta_abs_signed_gap_mean=("delta_abs_signed_gap_vs_source_ts", "mean"),

        nll_mean=("nll", "mean"),
        brier_mean=("brier", "mean"),
        ece15_mean=("ece15", "mean"),
        aece15_mean=("aece15", "mean"),
        signed_gap_mean=("signed_gap", "mean"),
        abs_signed_gap_mean=("abs_signed_gap", "mean"),

        source_ts_nll_mean=("source_ts_nll", "mean"),
        source_ts_brier_mean=("source_ts_brier", "mean"),
        source_ts_ece15_mean=("source_ts_ece15", "mean"),
        source_ts_aece15_mean=("source_ts_aece15", "mean"),

        oracle_T_mean=("oracle_T", "mean"),
        oracle_T_min=("oracle_T", "min"),
        oracle_T_max=("oracle_T", "max"),
        source_T_mean=("source_T", "mean"),

        n_conditions=("delta_nll_vs_source_ts", "count"),
    )
)

target_oracle_delta_summary_path = (
    DIRS["target_oracle_diagnostics"] / "stage4_vit_target_oracle_delta_summary.csv"
)

target_oracle_delta_summary.to_csv(target_oracle_delta_summary_path, index=False)

print("Saved target-oracle diagnostic summary:")
print(target_oracle_delta_summary_path)

display(target_oracle_delta_summary.round(6))

**Oracle temperature by severity**

In [ ]:
target_oracle_by_severity = (
    target_oracle_delta_wide
    .groupby(["architecture", "severity"], as_index=False)
    .agg(
        oracle_T_mean=("oracle_T", "mean"),
        oracle_T_std=("oracle_T", "std"),
        oracle_T_min=("oracle_T", "min"),
        oracle_T_max=("oracle_T", "max"),
        source_T_mean=("source_T", "mean"),

        delta_nll_mean=("delta_nll_vs_source_ts", "mean"),
        delta_brier_mean=("delta_brier_vs_source_ts", "mean"),
        delta_ece15_mean=("delta_ece15_vs_source_ts", "mean"),
        delta_aece15_mean=("delta_aece15_vs_source_ts", "mean"),
        delta_abs_signed_gap_mean=("delta_abs_signed_gap_vs_source_ts", "mean"),

        n_conditions=("condition", "count"),
    )
    .sort_values("severity")
)

target_oracle_by_severity_path = (
    DIRS["target_oracle_diagnostics"] / "stage4_vit_target_oracle_temperature_by_severity.csv"
)

target_oracle_by_severity.to_csv(target_oracle_by_severity_path, index=False)

print("Saved target-oracle by severity:")
print(target_oracle_by_severity_path)

display(target_oracle_by_severity.round(6))

**Oracle temperature by corruption**

In [ ]:
target_oracle_by_corruption = (
    target_oracle_delta_wide
    .groupby(["architecture", "corruption"], as_index=False)
    .agg(
        oracle_T_mean=("oracle_T", "mean"),
        oracle_T_std=("oracle_T", "std"),
        oracle_T_min=("oracle_T", "min"),
        oracle_T_max=("oracle_T", "max"),
        source_T_mean=("source_T", "mean"),

        delta_nll_mean=("delta_nll_vs_source_ts", "mean"),
        delta_brier_mean=("delta_brier_vs_source_ts", "mean"),
        delta_ece15_mean=("delta_ece15_vs_source_ts", "mean"),
        delta_aece15_mean=("delta_aece15_vs_source_ts", "mean"),
        delta_abs_signed_gap_mean=("delta_abs_signed_gap_vs_source_ts", "mean"),

        n_conditions=("condition", "count"),
    )
    .sort_values("oracle_T_mean", ascending=False)
)

target_oracle_by_corruption_path = (
    DIRS["target_oracle_diagnostics"] / "stage4_vit_target_oracle_temperature_by_corruption.csv"
)

target_oracle_by_corruption.to_csv(target_oracle_by_corruption_path, index=False)

print("Saved target-oracle by corruption:")
print(target_oracle_by_corruption_path)

display(target_oracle_by_corruption.round(6))

**Oracle family × severity table**

In [ ]:
target_oracle_by_corruption_severity = (
    target_oracle_delta_wide
    .groupby(["architecture", "corruption", "severity"], as_index=False)
    .agg(
        oracle_T_mean=("oracle_T", "mean"),
        oracle_T_std=("oracle_T", "std"),
        oracle_T_min=("oracle_T", "min"),
        oracle_T_max=("oracle_T", "max"),
        source_T_mean=("source_T", "mean"),

        delta_nll_mean=("delta_nll_vs_source_ts", "mean"),
        delta_brier_mean=("delta_brier_vs_source_ts", "mean"),
        delta_ece15_mean=("delta_ece15_vs_source_ts", "mean"),
        delta_aece15_mean=("delta_aece15_vs_source_ts", "mean"),
        delta_abs_signed_gap_mean=("delta_abs_signed_gap_vs_source_ts", "mean"),

        n_conditions=("condition", "count"),
    )
    .sort_values(["corruption", "severity"])
)

target_oracle_by_corruption_severity_path = (
    DIRS["target_oracle_diagnostics"] / "stage4_vit_target_oracle_temperature_by_corruption_severity.csv"
)

target_oracle_by_corruption_severity.to_csv(
    target_oracle_by_corruption_severity_path,
    index=False,
)

print("Saved target-oracle by corruption × severity:")
print(target_oracle_by_corruption_severity_path)

display(target_oracle_by_corruption_severity.round(6).head(30))

**Save Stage 4 completion manifest**

In [ ]:
stage4_manifest = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "architecture": ARCH_NAME,
    "stage": "stage4_target_oracle_diagnostics",
    "important_note": (
        "Target-oracle temperatures are diagnostic only because they use target labels. "
        "They must not be described as deployable calibration methods."
    ),
    "outputs": {
        "target_oracle_temperature_by_condition": str(target_oracle_temperature_path),
        "target_oracle_eval_long": str(target_oracle_eval_path),
        "target_oracle_condition_wide": str(target_oracle_condition_wide_path),
        "target_oracle_delta_vs_source_ts_wide": str(target_oracle_delta_wide_path),
        "target_oracle_delta_summary": str(target_oracle_delta_summary_path),
        "target_oracle_by_severity": str(target_oracle_by_severity_path),
        "target_oracle_by_corruption": str(target_oracle_by_corruption_path),
        "target_oracle_by_corruption_severity": str(target_oracle_by_corruption_severity_path),
    },
    "checks": {
        "uses_target_labels": True,
        "diagnostic_only": True,
        "temperature_positive": bool(target_oracle_temperature_df["oracle_T"].gt(0).all()),
        "oracle_fits_do_not_worsen_nll": bool(len(bad_oracle_fits) == 0),
        "ece15_values_between_0_and_1": bool(target_oracle_condition_wide["ece15"].between(0, 1).all()),
        "aece15_values_between_0_and_1": bool(target_oracle_condition_wide["aece15"].between(0, 1).all()),
        "accuracy_preserved_vs_source_ts": bool(
            (target_oracle_delta_wide["delta_accuracy_vs_source_ts"].abs() < 1e-10).all()
        ),
    },
}

stage4_manifest_path = (
    DIRS["manifests"] / "stage4_vit_target_oracle_diagnostics_manifest.json"
)

with open(stage4_manifest_path, "w") as f:
    json.dump(stage4_manifest, f, indent=2)

print("Saved Stage 4 manifest:")
print(stage4_manifest_path)

print("\n========== STAGE 4 DONE ==========")
print("Target-oracle summary:", target_oracle_delta_summary_path)
print("Oracle by severity:", target_oracle_by_severity_path)
print("Oracle by corruption:", target_oracle_by_corruption_path)

display(target_oracle_delta_summary.round(6))
display(target_oracle_by_severity.round(6))
display(target_oracle_by_corruption.round(6))

**Stage 5.1 — Install/import corruption library**

In [ ]:
!pip -q install imagecorruptions

import numpy as np
from PIL import Image
from imagecorruptions import corrupt

print("imagecorruptions imported successfully.")

**Source-bank configuration**

In [ ]:
SOURCE_BANK_N_SAMPLES = 4096

SOURCE_BANK_ROOT = VIT_EXTRA_ROOT / "source_bank_logits"
SOURCE_BANK_LOGITS_DIR = SOURCE_BANK_ROOT / ARCH_NAME

SOURCE_BANK_ROOT.mkdir(parents=True, exist_ok=True)
SOURCE_BANK_LOGITS_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_BANK_CONFIG = {
    "architecture": ARCH_NAME,
    "source_bank_n_samples": SOURCE_BANK_N_SAMPLES,
    "source_data": "clean ImageNet-100 training split",
    "corruption_library": "imagecorruptions",
    "corruptions": CORRUPTIONS,
    "severities": SEVERITIES,
    "seeds": SEEDS,
    "important_note": (
        "Source-bank logits are generated from clean ImageNet-100 source images with synthetic corruptions. "
        "They do not use target ImageNet-C images for fitting stream methods."
    ),
}

source_bank_config_path = DIRS["configs"] / "stage5_vit_source_bank_config.json"

with open(source_bank_config_path, "w") as f:
    json.dump(SOURCE_BANK_CONFIG, f, indent=2)

print("Source-bank logits dir:")
print(SOURCE_BANK_LOGITS_DIR)

print("\nSaved source-bank config:")
print(source_bank_config_path)

**Clean source dataset without transform**

In [ ]:
raw_source_dataset = datasets.ImageFolder(
    root=str(TRAIN_IMAGEFOLDER_ROOT),
    transform=None,
)

assert raw_source_dataset.classes == sorted(IMAGENET100_SYNSETS), (
    "Raw source dataset class order mismatch."
)

print("Raw source dataset size:", len(raw_source_dataset))
print("Num classes:", len(raw_source_dataset.classes))
print("First 10 classes:", raw_source_dataset.classes[:10])

**Balanced source-bank sampler**

In [ ]:
def make_balanced_source_bank_indices(seed: int, n_samples: int = 4096):
    """
    Sample source-bank indices from the clean training split only.
    The sample is approximately balanced across classes.
    """
    rng = np.random.default_rng(seed + 5000)

    train_targets = np.array(raw_source_dataset.targets)
    train_index_set = set(train_indices.tolist())

    class_to_available_indices = {}

    for class_id in range(NUM_CLASSES):
        cls_indices = [
            idx for idx in train_indices.tolist()
            if train_targets[idx] == class_id
        ]

        assert len(cls_indices) > 0, f"No training images for class {class_id}"

        rng.shuffle(cls_indices)
        class_to_available_indices[class_id] = cls_indices

    base_per_class = n_samples // NUM_CLASSES
    remainder = n_samples % NUM_CLASSES

    selected = []

    for class_id in range(NUM_CLASSES):
        take = base_per_class + (1 if class_id < remainder else 0)
        available = class_to_available_indices[class_id]

        if len(available) >= take:
            chosen = available[:take]
        else:
            chosen = rng.choice(available, size=take, replace=True).tolist()

        selected.extend(chosen)

    rng.shuffle(selected)

    assert len(selected) == n_samples

    selected_targets = train_targets[selected]
    unique_classes = set(selected_targets.tolist())

    assert len(unique_classes) == NUM_CLASSES, (
        f"Source-bank sample does not cover all classes. Covered {len(unique_classes)}"
    )

    return selected


source_bank_indices_by_seed = {}

for seed in SEEDS:
    idxs = make_balanced_source_bank_indices(seed, SOURCE_BANK_N_SAMPLES)
    source_bank_indices_by_seed[int(seed)] = idxs

    labels = np.array(raw_source_dataset.targets)[idxs]

    print(
        f"seed={seed} | n={len(idxs)} | "
        f"classes={len(set(labels.tolist()))} | "
        f"min_per_class={np.bincount(labels, minlength=NUM_CLASSES).min()} | "
        f"max_per_class={np.bincount(labels, minlength=NUM_CLASSES).max()}"
    )

source_bank_indices_manifest = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "source_bank_n_samples": SOURCE_BANK_N_SAMPLES,
    "indices_by_seed": {str(k): v for k, v in source_bank_indices_by_seed.items()},
}

source_bank_indices_path = DIRS["manifests"] / "stage5_vit_source_bank_indices_by_seed.json"

with open(source_bank_indices_path, "w") as f:
    json.dump(source_bank_indices_manifest, f, indent=2)

print("\nSaved source-bank indices:")
print(source_bank_indices_path)

**Corrupted source-bank dataset class**

In [ ]:
pre_corrupt_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
])

post_corrupt_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class CorruptedSourceBankDataset(torch.utils.data.Dataset):
    """
    Source-bank pseudo-domain dataset.

    Loads clean ImageNet-100 source images, applies a chosen ImageNet-C-style
    corruption using imagecorruptions, then applies ViT normalization.
    """

    def __init__(self, base_dataset, selected_indices, corruption_name: str, severity: int):
        self.base_dataset = base_dataset
        self.selected_indices = list(selected_indices)
        self.corruption_name = corruption_name
        self.severity = int(severity)

        assert self.corruption_name in CORRUPTIONS
        assert self.severity in SEVERITIES

    def __len__(self):
        return len(self.selected_indices)

    def __getitem__(self, i):
        base_idx = self.selected_indices[i]

        img, label = self.base_dataset[base_idx]

        if img.mode != "RGB":
            img = img.convert("RGB")

        # Match eval size before corruption.
        img = pre_corrupt_transform(img)

        img_np = np.array(img).astype(np.uint8)

        corrupted_np = corrupt(
            img_np,
            corruption_name=self.corruption_name,
            severity=self.severity,
        )

        corrupted_np = np.clip(corrupted_np, 0, 255).astype(np.uint8)

        corrupted_img = Image.fromarray(corrupted_np)

        x = post_corrupt_transform(corrupted_img)

        return x, int(label)


# Quick test
_test_bank_dataset = CorruptedSourceBankDataset(
    base_dataset=raw_source_dataset,
    selected_indices=source_bank_indices_by_seed[SEEDS[0]][:16],
    corruption_name="gaussian_noise",
    severity=1,
)

x0, y0 = _test_bank_dataset[0]

print("Test tensor shape:", x0.shape)
print("Test label:", y0)
print("Test dataset length:", len(_test_bank_dataset))

del _test_bank_dataset

**Cache source-bank logits helper**

In [ ]:
@torch.no_grad()
def cache_source_bank_logits(model, loader, device):
    model.eval()

    all_logits = []
    all_labels = []

    total_correct = 0
    total_nll = 0.0
    total_samples = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = F.cross_entropy(logits, y, reduction="sum")

        pred = logits.argmax(dim=1)

        total_correct += pred.eq(y).sum().item()
        total_nll += loss.item()
        total_samples += y.size(0)

        all_logits.append(logits.detach().cpu())
        all_labels.append(y.detach().cpu())

    logits = torch.cat(all_logits, dim=0)
    labels = torch.cat(all_labels, dim=0)

    return {
        "logits": logits.float(),
        "labels": labels.long(),
        "accuracy": float(total_correct / max(total_samples, 1)),
        "nll": float(total_nll / max(total_samples, 1)),
        "num_samples": int(total_samples),
    }


def source_bank_logits_path(seed: int, corruption: str, severity: int):
    condition = f"{corruption}_sev{severity}"
    return SOURCE_BANK_LOGITS_DIR / f"seed{seed}_{condition}_n{SOURCE_BANK_N_SAMPLES}.pt"


def save_source_bank_logits(path, result, metadata):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    torch.save(
        {
            "logits": result["logits"],
            "labels": result["labels"],
            "metadata": metadata,
        },
        path,
    )


print("Source-bank caching helpers ready.")

**Cache ViT source-bank logits for all seeds/conditions**

In [ ]:
# ============================================================
# Compatibility patches for imagecorruptions with NumPy 2.x
# ============================================================

import numpy as np

# imagecorruptions uses np.float_, removed in NumPy 2.0
if not hasattr(np, "float_"):
    np.float_ = np.float64

if not hasattr(np, "int"):
    np.int = int

if not hasattr(np, "float"):
    np.float = float

print("Patched NumPy aliases for imagecorruptions.")

In [ ]:
import imagecorruptions.corruptions as ic_corr
from skimage.filters import gaussian as skimage_gaussian

def gaussian_compat(image, sigma=1, multichannel=None, channel_axis=None, **kwargs):
    """
    Compatibility wrapper for imagecorruptions:
    old API: gaussian(..., multichannel=True)
    new API: gaussian(..., channel_axis=-1)
    """
    if multichannel is not None:
        channel_axis = -1 if multichannel else None

    return skimage_gaussian(
        image,
        sigma=sigma,
        channel_axis=channel_axis,
        **kwargs,
    )

ic_corr.gaussian = gaussian_compat
print("Patched imagecorruptions.gaussian for newer scikit-image.")

In [ ]:
from tqdm.auto import tqdm

stage5_rows = []

for seed in SEEDS:
    print(f"\n{'='*90}")
    print(f"Computing source-bank logits | {ARCH_NAME} | seed={seed}")
    print(f"{'='*90}")

    model = None
    ckpt = None
    checkpoint_path = None

    selected_indices = source_bank_indices_by_seed[int(seed)]

    for corruption in CORRUPTIONS:
        for severity in SEVERITIES:
            condition = f"{corruption}_sev{severity}"
            output_path = source_bank_logits_path(seed, corruption, severity)

            if output_path.exists():
                print(f"[SKIP] {output_path.name} (already cached)")

                blob = torch.load(output_path, map_location="cpu", weights_only=False)
                logits = blob["logits"].float()
                labels = blob["labels"].long()
                raw_acc = logits.argmax(dim=1).eq(labels).float().mean().item()
                raw_nll = F.cross_entropy(logits, labels, reduction="mean").item()

                meta = blob.get("metadata", {}) if isinstance(blob, dict) else {}
                cached_ckpt = meta.get("checkpoint", "") if isinstance(meta, dict) else ""

                stage5_rows.append({
                    "architecture": ARCH_NAME,
                    "seed": int(seed),
                    "corruption": corruption,
                    "severity": int(severity),
                    "condition": condition,
                    "checkpoint": cached_ckpt,
                    "source_bank_logits_path": str(output_path),
                    "num_samples": int(len(labels)),
                    "accuracy": float(raw_acc),
                    "nll": float(raw_nll),
                    "status": "skipped_existing",
                })

                del blob, logits, labels
                continue


            if model is None:
                model, ckpt, checkpoint_path = load_vit_best_by_nll(seed)

            print(f"\n[RUN] seed={seed} {condition}")

            bank_dataset = CorruptedSourceBankDataset(
                base_dataset=raw_source_dataset,
                selected_indices=selected_indices,
                corruption_name=corruption,
                severity=severity,
            )

            bank_loader = DataLoader(
                bank_dataset,
                batch_size=TRAIN_CONFIG["batch_size"],
                shuffle=False,
                num_workers=4,
                pin_memory=True,
                persistent_workers=False,
            )

            model.eval()

            all_logits = []
            all_labels = []
            total_correct = 0
            total_nll = 0.0
            total_samples = 0

            with torch.no_grad():
                for x, y in tqdm(bank_loader, desc=f"{condition}", leave=False):
                    x = x.to(DEVICE, non_blocking=True)
                    y = y.to(DEVICE, non_blocking=True)

                    logits = model(x)
                    loss = F.cross_entropy(logits, y, reduction="sum")

                    pred = logits.argmax(dim=1)

                    total_correct += pred.eq(y).sum().item()
                    total_nll += loss.item()
                    total_samples += y.size(0)

                    all_logits.append(logits.detach().cpu())
                    all_labels.append(y.detach().cpu())

            result = {
                "logits": torch.cat(all_logits, dim=0).float(),
                "labels": torch.cat(all_labels, dim=0).long(),
                "accuracy": float(total_correct / max(total_samples, 1)),
                "nll": float(total_nll / max(total_samples, 1)),
                "num_samples": int(total_samples),
            }

            metadata = {
                "architecture": ARCH_NAME,
                "seed": int(seed),
                "checkpoint": str(checkpoint_path),
                "source_bank_n_samples": SOURCE_BANK_N_SAMPLES,
                "source_data": "clean ImageNet-100 train split",
                "corruption": corruption,
                "severity": int(severity),
                "condition": condition,
                "corruption_library": "imagecorruptions",
                "num_samples": int(result["num_samples"]),
                "accuracy": float(result["accuracy"]),
                "nll": float(result["nll"]),
                "classes": IMAGENET100_SYNSETS,
                "class_to_idx": class_to_idx,
                "created_at": datetime.now().isoformat(timespec="seconds"),
            }

            save_source_bank_logits(output_path, result, metadata)

            stage5_rows.append({
                "architecture": ARCH_NAME,
                "seed": int(seed),
                "corruption": corruption,
                "severity": int(severity),
                "condition": condition,
                "checkpoint": str(checkpoint_path),
                "source_bank_logits_path": str(output_path),
                "num_samples": int(result["num_samples"]),
                "accuracy": float(result["accuracy"]),
                "nll": float(result["nll"]),
                "status": "computed",
            })

            print(
                f"[DONE] {condition:25s} | "
                f"n={result['num_samples']:5d} | "
                f"acc={result['accuracy']:.4f} | "
                f"nll={result['nll']:.4f}"
            )

            del bank_dataset, bank_loader, all_logits, all_labels, result
            torch.cuda.empty_cache()

    # Free model after all conditions for this seed
    if model is not None:
        del model
        torch.cuda.empty_cache()

stage5_source_bank_manifest_df = pd.DataFrame(stage5_rows)

stage5_source_bank_manifest_path = (
    DIRS["manifests"] / "stage5_vit_source_bank_logits_manifest.csv"
)

stage5_source_bank_manifest_df.to_csv(stage5_source_bank_manifest_path, index=False)

print("\nSaved Stage 5 source-bank logits manifest:")
print(stage5_source_bank_manifest_path)

print("\nStatus counts:")
print(stage5_source_bank_manifest_df["status"].value_counts())

display(stage5_source_bank_manifest_df.head())

**Validate all source-bank logits**

In [ ]:
expected_source_bank_files = len(SEEDS) * len(CORRUPTIONS) * len(SEVERITIES)

source_bank_files = sorted(SOURCE_BANK_LOGITS_DIR.glob(f"seed*_n{SOURCE_BANK_N_SAMPLES}.pt"))

print("Expected source-bank files:", expected_source_bank_files)
print("Found source-bank files:", len(source_bank_files))

assert len(source_bank_files) == expected_source_bank_files, (
    f"Expected {expected_source_bank_files} files, found {len(source_bank_files)}"
)

validation_rows = []

for seed in SEEDS:
    for corruption in CORRUPTIONS:
        for severity in SEVERITIES:
            condition = f"{corruption}_sev{severity}"
            path = source_bank_logits_path(seed, corruption, severity)

            assert path.exists(), f"Missing source-bank logits: {path}"

            blob = torch.load(path, map_location="cpu", weights_only=False)

            logits = blob["logits"].float()
            labels = blob["labels"].long()

            assert logits.ndim == 2, f"Bad logits shape: {logits.shape}"
            assert labels.ndim == 1, f"Bad labels shape: {labels.shape}"
            assert logits.shape[0] == labels.shape[0], "Logits/labels length mismatch"
            assert logits.shape[1] == NUM_CLASSES, f"Expected {NUM_CLASSES} classes"
            assert len(labels) == SOURCE_BANK_N_SAMPLES, (
                f"Expected {SOURCE_BANK_N_SAMPLES} samples, got {len(labels)}"
            )

            acc = logits.argmax(dim=1).eq(labels).float().mean().item()
            nll = F.cross_entropy(logits, labels, reduction="mean").item()

            validation_rows.append({
                "architecture": ARCH_NAME,
                "seed": int(seed),
                "corruption": corruption,
                "severity": int(severity),
                "condition": condition,
                "path": str(path),
                "num_samples": int(len(labels)),
                "num_classes": int(logits.shape[1]),
                "accuracy": float(acc),
                "nll": float(nll),
            })

stage5_source_bank_validation_df = pd.DataFrame(validation_rows)

stage5_source_bank_validation_path = (
    DIRS["tables"] / "stage5_vit_source_bank_logits_validation.csv"
)

stage5_source_bank_validation_df.to_csv(
    stage5_source_bank_validation_path,
    index=False,
)

print("Saved source-bank validation table:")
print(stage5_source_bank_validation_path)

display(stage5_source_bank_validation_df.head())

display(
    stage5_source_bank_validation_df
    .groupby(["seed"], as_index=False)
    .agg(
        n_conditions=("condition", "count"),
        total_samples=("num_samples", "sum"),
        mean_accuracy=("accuracy", "mean"),
        mean_nll=("nll", "mean"),
        min_accuracy=("accuracy", "min"),
        max_nll=("nll", "max"),
    )
    .round(6)
)

**Source-bank summary by corruption/severity**

In [ ]:
stage5_source_bank_by_corruption = (
    stage5_source_bank_validation_df
    .groupby(["architecture", "corruption"], as_index=False)
    .agg(
        mean_accuracy=("accuracy", "mean"),
        mean_nll=("nll", "mean"),
        n_conditions=("condition", "count"),
        total_samples=("num_samples", "sum"),
    )
    .sort_values("mean_nll", ascending=False)
)

stage5_source_bank_by_severity = (
    stage5_source_bank_validation_df
    .groupby(["architecture", "severity"], as_index=False)
    .agg(
        mean_accuracy=("accuracy", "mean"),
        mean_nll=("nll", "mean"),
        n_conditions=("condition", "count"),
        total_samples=("num_samples", "sum"),
    )
    .sort_values("severity")
)

stage5_source_bank_by_corruption_path = (
    DIRS["tables"] / "stage5_vit_source_bank_by_corruption.csv"
)

stage5_source_bank_by_severity_path = (
    DIRS["tables"] / "stage5_vit_source_bank_by_severity.csv"
)

stage5_source_bank_by_corruption.to_csv(
    stage5_source_bank_by_corruption_path,
    index=False,
)

stage5_source_bank_by_severity.to_csv(
    stage5_source_bank_by_severity_path,
    index=False,
)

print("Saved source-bank by-corruption summary:")
print(stage5_source_bank_by_corruption_path)

print("\nSaved source-bank by-severity summary:")
print(stage5_source_bank_by_severity_path)

print("\nSource-bank by severity:")
display(stage5_source_bank_by_severity.round(6))

print("\nSource-bank by corruption:")
display(stage5_source_bank_by_corruption.round(6))

**Save Stage 5 completion manifest**

In [ ]:
stage5_manifest = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "architecture": ARCH_NAME,
    "stage": "stage5_build_vit_source_bank_logits",
    "source_bank_root": str(SOURCE_BANK_ROOT),
    "source_bank_logits_dir": str(SOURCE_BANK_LOGITS_DIR),
    "source_bank_n_samples": SOURCE_BANK_N_SAMPLES,
    "source_data": "clean ImageNet-100 train split",
    "corruption_library": "imagecorruptions",
    "seeds": SEEDS,
    "corruptions": CORRUPTIONS,
    "severities": SEVERITIES,
    "expected_num_files": expected_source_bank_files,
    "actual_num_files": len(source_bank_files),
    "outputs": {
        "source_bank_config": str(source_bank_config_path),
        "source_bank_indices": str(source_bank_indices_path),
        "source_bank_logits_manifest": str(stage5_source_bank_manifest_path),
        "source_bank_validation": str(stage5_source_bank_validation_path),
        "source_bank_by_corruption": str(stage5_source_bank_by_corruption_path),
        "source_bank_by_severity": str(stage5_source_bank_by_severity_path),
    },
    "checks": {
        "uses_clean_source_images_only": True,
        "does_not_use_target_imagenetc_for_source_bank_fitting": True,
        "num_files_complete": bool(len(source_bank_files) == expected_source_bank_files),
        "all_logits_have_100_classes": bool((stage5_source_bank_validation_df["num_classes"] == NUM_CLASSES).all()),
        "all_files_have_expected_sample_count": bool(
            (stage5_source_bank_validation_df["num_samples"] == SOURCE_BANK_N_SAMPLES).all()
        ),
    },
    "important_note": (
        "These source-bank logits are used to fit entropy/ridge/V2/V3 stream calibration methods. "
        "Target ImageNet-C labels are not used for this source-bank generation."
    ),
}

stage5_manifest_path = DIRS["manifests"] / "stage5_vit_source_bank_logits_manifest.json"

with open(stage5_manifest_path, "w") as f:
    json.dump(stage5_manifest, f, indent=2)

print("Saved Stage 5 completion manifest:")
print(stage5_manifest_path)

print("\n========== STAGE 5 DONE ==========")
print("Source-bank logits dir:", SOURCE_BANK_LOGITS_DIR)
print("Expected files:", expected_source_bank_files)
print("Actual files:", len(source_bank_files))
print("Validation table:", stage5_source_bank_validation_path)

display(stage5_source_bank_validation_df.head())
display(stage5_source_bank_by_severity.round(6))